# Planner + Caller — evidence, intent and throughput
T4 x2, Internet on. Two independent local QLoRA adapters from the original Qwen3-4B, one epoch each. Choose STAGES to train either role alone. No Global, Summarizer, ASR or automatic workflow inference. No initial smoke run.
English prompts and TypeScript contracts. Evidence-grounded supervision includes conditions, observed entities, times and pending goals; richer plans may contain justified values. Training-only repair examples teach argument completeness and reuse of existing reads.100 training no-action targets were manually reviewed; all other rationales are a transparent Codex-authored evidence compiler, not independently generated per-example teacher reasoning. Split IDs remain1300/245; suspect training episodes are quarantined with a report, never silently repaired from hidden state. Validation no-action route targets are omitted because contest annotations do not identify the route.
Unsloth compile enabled, FP16/4-bit base, rank16 alpha32, microbatch train/eval4 per GPU, effective batch8, cosine LR/warmup, patience3. Assistant-token CE is unchanged mathematically: projection chunk512, projection checkpoint OFF for throughput, backbone checkpoint ON for long T4 batches. This costs more VRAM and is not yet GPU-benchmarked. No evidence is truncated. Tokenization is cached once; validation is length-sorted. Initial CE and repeated CE after hash-verified best restoration are skipped by default. Logs separate train throughput, padding, eval, checkpoint I/O and peak VRAM.
Save every25 attempted steps, after eval and cooperative deadline. Preserve the WHOLE WORK directory and attach it to resume. Hard cancellation can lose updates since the last checkpoint. Old adapters/runs are incompatible with this new prompt/data version.
After each completed role, free training GPUs and run vLLM generation evaluation for CE-best and last checkpoints on the same reference-prefix inputs, batch4/GPU, no enforcer. Report route/tool agreement, terminal errors, action match and per-field/argument metrics. Recommend one of those two candidates; do not call it a full-pipeline contest score. Review exported rationales and error cases before promoting adapters.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,zipfile,shutil,time
STAGES=('planner','caller')  # ('planner',) or ('caller',) trains only that node
EPOCHS={'planner':1,'caller':1}
RUN_NAME='planner_caller_evidence_v3'
RESUME_DIR=None  # Previous entire WORK directory mounted as a private dataset
RUN_NODE_EVAL=True
LOSS_CHUNK=512
PROJECTION_CHECKPOINT=False  # More VRAM, less backward recomputation
EVAL_STEPS=100
SAVE_STEPS=25
WORK=Path('/kaggle/working/planner_caller_evidence')/RUN_NAME
WORK.mkdir(parents=True,exist_ok=True)
if RESUME_DIR is not None:shutil.copytree(Path(RESUME_DIR),WORK,dirs_exist_ok=True)
SOURCE=Path('/kaggle/working/planner_caller_evidence_source')
MODEL='Qwen/Qwen3-4B-Instruct-2507';REVISION='cdbee75f17c01a7cc42f958dc650907174af0554'
DEADLINE=time.time()+10*3600
assert STAGES and len(set(STAGES))==len(STAGES) and set(STAGES)<={'planner','caller'}
if not SOURCE.exists():subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/tnisl/baseline.git',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin','4894d331692293565635454a8c686b7c18314a04'],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)


In [ ]:
PAYLOAD={'requirements_infer_2gpu.lock': '--extra-index-url https://download.pytorch.org/whl/cu128\naccelerate==1.15.0\nagent-detector==2.0.0\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nairportsdata==20261030\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.16.0\nastor==0.8.1\nattrs==26.1.0\nblake3==1.0.11\ncachetools==7.2.8\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ncompressed-tensors==0.10.2\ncuda-pathfinder==1.8.5\ncupy-cuda12x==14.2.0\ndepyf==0.18.0\ndetect-installer==0.2.1\ndill==0.4.1\ndiskcache==5.6.3\ndistro==1.9.0\ndnspython==2.9.1\neinops==0.8.2\nemail-validator==2.3.0\nfastapi==0.162.0\nfastapi-cli==0.0.32\nfastapi-cloud-cli==0.26.0\nfastar==0.12.0\nfilelock==4.2.7\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngguf==0.19.0\ngoogleapis-common-protos==1.76.0\nh11==0.16.0\nhf-xet==1.8.0\nhttpcore==1.0.9\nhttpcore2==2.16.0\nhttptools==0.9.0\nhttpx==0.28.1\nhttpx2==2.16.0\nhuggingface-hub==0.35.3\nidna==3.20\ninteregular==0.3.3\njinja2==3.1.6\njiter==0.18.0\njsonschema==4.25.1\njsonschema-specifications==2025.9.1\nlark==1.2.2\nllguidance==0.7.30\nllvmlite==0.44.0\nlm-format-enforcer==0.10.12\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmistral-common==1.13.1\nmpmath==1.3.0\nmsgpack==1.2.4\nmsgspec==0.22.0\nmultidict==7.2.1\nnest-asyncio==1.6.0\nnetworkx==3.7\nninja==1.13.2\nnumba==0.61.2\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.3.14\nnvidia-cuda-cupti-cu12==12.8.57\nnvidia-cuda-nvrtc-cu12==12.8.61\nnvidia-cuda-runtime-cu12==12.8.57\nnvidia-cudnn-cu12==9.7.1.26\nnvidia-cufft-cu12==11.3.3.41\nnvidia-cufile-cu12==1.13.0.11\nnvidia-curand-cu12==10.3.9.55\nnvidia-cusolver-cu12==11.7.2.55\nnvidia-cusparse-cu12==12.5.7.53\nnvidia-cusparselt-cu12==0.6.3\nnvidia-nccl-cu12==2.26.2\nnvidia-nvjitlink-cu12==12.8.61\nnvidia-nvtx-cu12==12.8.55\nopenai==1.90.0\nopencv-python-headless==5.0.0.93\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\noutlines==0.1.11\noutlines-core==0.1.26\npackaging==26.3\npartial-json-parser==0.2.1.1.post7\npeft==0.17.1\npillow==13.0.0\nprometheus-client==0.26.0\nprometheus-fastapi-instrumentator==8.1.0\npropcache==0.5.4\nprotobuf==7.36.2\npsutil==7.2.2\npy-cpuinfo==9.0.0\npybase64==1.5.1\npycountry==26.2.16\npydantic==2.14.2\npydantic-core==2.50.2\npydantic-extra-types==2.11.1\npydantic-settings==2.16.2\npygments==2.21.0\npython-dotenv==1.2.4\npython-json-logger==4.2.0\npython-multipart==0.0.33\npyyaml==6.0.3\npyzmq==27.2.0\nray==2.59.0\nreferencing==0.37.0\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nrich-toolkit==0.20.6\nrignore==0.8.1\nrpds-py==2026.9.1\nsafetensors==0.8.0\nscipy==1.18.1\nsentencepiece==0.2.2\nsentry-sdk==2.73.0\nsetuptools==79.0.1\nshellingham==1.5.4\nsix==1.17.0\nsniffio==1.3.1\nstarlette==1.10.0\nsympy==1.14.0\ntiktoken==0.14.0\ntokenizers==0.21.4\ntorch==2.7.0+cu128\ntorchaudio==2.7.0+cu128\ntorchvision==0.22.0+cu128\ntqdm==4.70.1\ntransformers==4.53.3\ntriton==3.3.0\ntruststore==0.10.5\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\nurllib3==2.8.0\nuvicorn==0.54.0\nuvloop==0.23.0\nvllm==0.9.2\nwatchfiles==1.3.0\nwebsockets==17.4\nwrapt==1.17.3\nxformers==0.0.30\nxgrammar==0.1.19\nyarl==1.25.1\n', 'requirements_train_2gpu.lock': 'accelerate==1.10.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\nagent-detector==2.0.0\n    # via fastapi-cloud-cli\naiohappyeyeballs==2.7.1\n    # via aiohttp\naiohttp==3.14.5\n    # via\n    #   fsspec\n    #   vllm\naiosignal==1.4.0\n    # via aiohttp\nannotated-doc==0.0.5\n    # via\n    #   fastapi\n    #   typer\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\n    #   openai\n    #   starlette\n    #   watchfiles\nastor==0.8.1\n    # via depyf\nattrs==26.1.0\n    # via\n    #   aiohttp\n    #   jsonschema\n    #   referencing\nbitsandbytes==0.47.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nblake3==1.0.11\n    # via vllm\ncachetools==7.2.8\n    # via vllm\ncbor2==6.2.0.post2\n    # via vllm\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\n    #   sentry-sdk\ncffi==2.1.1\n    # via soundfile\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via\n    #   ray\n    #   rich-toolkit\n    #   uvicorn\ncloudpickle==3.1.2\n    # via vllm\ncompressed-tensors==0.11.0\n    # via vllm\ncuda-pathfinder==1.8.5\n    # via cupy-cuda12x\ncupy-cuda12x==14.2.0\n    # via ray\ncut-cross-entropy==25.1.1\n    # via unsloth-zoo\ndatasets==3.6.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\ndepyf==0.19.0\n    # via vllm\ndetect-installer==0.2.1\n    # via fastapi-cloud-cli\ndiffusers==0.35.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\ndill==0.3.8\n    # via\n    #   datasets\n    #   depyf\n    #   multiprocess\ndiskcache==5.6.3\n    # via vllm\ndnspython==2.9.1\n    # via email-validator\ndocstring-parser==0.18.0\n    # via tyro\neinops==0.8.2\n    # via vllm\nemail-validator==2.3.0\n    # via\n    #   fastapi\n    #   pydantic\nfastapi==0.162.0\n    # via vllm\nfastapi-cli==0.0.32\n    # via fastapi\nfastapi-cloud-cli==0.26.0\n    # via fastapi-cli\nfastar==0.12.0\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nfilelock==4.2.7\n    # via\n    #   datasets\n    #   diffusers\n    #   huggingface-hub\n    #   ray\n    #   torch\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nfrozendict==2.5.1\n    # via compressed-tensors\nfrozenlist==1.8.0\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2025.3.0\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   torch\ngguf==0.19.0\n    # via vllm\ngoogleapis-common-protos==1.76.0\n    # via opentelemetry-exporter-otlp-proto-http\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nhf-transfer==0.1.9\n    # via\n    #   unsloth\n    #   unsloth-zoo\nhf-xet==1.8.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttptools==0.9.0\n    # via uvicorn\nhttpx==0.28.1\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nhttpx2==2.16.0\n    # via\n    #   fastapi\n    #   openai\nhuggingface-hub==0.35.3\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   datasets\n    #   diffusers\n    #   peft\n    #   tokenizers\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\nidna==3.20\n    # via\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimportlib-metadata==10.1.0\n    # via diffusers\ninteregular==0.3.3\n    # via lm-format-enforcer\njinja2==3.1.6\n    # via\n    #   fastapi\n    #   torch\njiter==0.18.0\n    # via openai\njsonschema==4.26.0\n    # via\n    #   mistral-common\n    #   ray\njsonschema-specifications==2025.9.1\n    # via jsonschema\nlark==1.2.2\n    # via vllm\nllguidance==0.7.30\n    # via vllm\nllvmlite==0.44.0\n    # via numba\nlm-format-enforcer==0.11.3\n    # via vllm\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmdurl==0.1.2\n    # via markdown-it-py\nmistral-common==1.13.1\n    # via vllm\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via ray\nmsgspec==0.22.0\n    # via\n    #   unsloth-zoo\n    #   vllm\nmultidict==7.2.1\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.16\n    # via datasets\nnetworkx==3.7\n    # via torch\nninja==1.13.2\n    # via\n    #   vllm\n    #   xgrammar\nnumba==0.61.2\n    # via vllm\nnumpy==2.2.6\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   cupy-cuda12x\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   mistral-common\n    #   numba\n    #   opencv-python-headless\n    #   pandas\n    #   peft\n    #   scipy\n    #   soundfile\n    #   soxr\n    #   torchvision\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\nnvidia-cublas-cu12==12.8.4.1\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21\n    # via torch\nnvidia-cufft-cu12==11.3.3.83\n    # via torch\nnvidia-cufile-cu12==1.13.1.3\n    # via torch\nnvidia-curand-cu12==10.3.9.90\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1\n    # via torch\nnvidia-nccl-cu12==2.27.3\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.8.90\n    # via torch\nopenai==3.42.1\n    # via vllm\nopenai-harmony==0.0.8\n    # via vllm\nopencv-python-headless==5.0.0.93\n    # via\n    #   mistral-common\n    #   vllm\nopentelemetry-api==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\nopentelemetry-exporter-http-transport==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-common==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-common==1.45.2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-http==1.45.2\n    # via fastapi\nopentelemetry-proto==1.45.2\n    # via\n    #   opentelemetry-exporter-otlp-proto-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-sdk==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-otlp-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-semantic-conventions==0.66b2\n    # via opentelemetry-sdk\noutlines-core==0.2.11\n    # via vllm\npackaging==26.3\n    # via\n    #   accelerate\n    #   datasets\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   wheel\npandas==3.1.0\n    # via datasets\npartial-json-parser==0.2.1.1.post7\n    # via vllm\npeft==0.17.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\npillow==13.0.0\n    # via\n    #   diffusers\n    #   mistral-common\n    #   torchvision\n    #   unsloth-zoo\n    #   vllm\nprometheus-client==0.26.0\n    # via\n    #   prometheus-fastapi-instrumentator\n    #   vllm\nprometheus-fastapi-instrumentator==8.1.0\n    # via vllm\npropcache==0.5.4\n    # via\n    #   aiohttp\n    #   yarl\nprotobuf==7.36.2\n    # via\n    #   googleapis-common-protos\n    #   opentelemetry-proto\n    #   ray\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npsutil==7.2.2\n    # via\n    #   accelerate\n    #   peft\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npy-cpuinfo==9.0.0\n    # via vllm\npyarrow==26.0.0\n    # via datasets\npybase64==1.5.1\n    # via vllm\npycountry==26.2.16\n    # via pydantic-extra-types\npycparser==3.12\n    # via cffi\npydantic==2.14.2\n    # via\n    #   compressed-tensors\n    #   fastapi\n    #   fastapi-cloud-cli\n    #   lm-format-enforcer\n    #   mistral-common\n    #   openai\n    #   openai-harmony\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   vllm\n    #   xgrammar\npydantic-core==2.50.2\n    # via pydantic\npydantic-extra-types==2.11.1\n    # via\n    #   fastapi\n    #   mistral-common\npydantic-settings==2.16.2\n    # via fastapi\npygments==2.21.0\n    # via rich\npython-dateutil==4.0.0\n    # via pandas\npython-dotenv==1.2.4\n    # via\n    #   pydantic-settings\n    #   uvicorn\npython-json-logger==4.2.0\n    # via vllm\npython-multipart==0.0.33\n    # via fastapi\npyyaml==6.0.3\n    # via\n    #   accelerate\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   uvicorn\n    #   vllm\npyzmq==27.2.0\n    # via vllm\nray==2.59.0\n    # via vllm\nreferencing==0.37.0\n    # via\n    #   jsonschema\n    #   jsonschema-specifications\nregex==2026.9.29\n    # via\n    #   diffusers\n    #   tiktoken\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nrequests==2.34.2\n    # via\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   huggingface-hub\n    #   mistral-common\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   ray\n    #   tiktoken\n    #   transformers\n    #   vllm\nrich==15.0.0\n    # via\n    #   rich-toolkit\n    #   typer\nrich-toolkit==0.20.6\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nrignore==0.8.1\n    # via fastapi-cloud-cli\nrpds-py==2026.9.1\n    # via\n    #   jsonschema\n    #   referencing\nsafetensors==0.8.0\n    # via\n    #   accelerate\n    #   diffusers\n    #   peft\n    #   transformers\nscipy==1.18.1\n    # via vllm\nsentencepiece==0.2.2\n    # via\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\nsentry-sdk==2.73.0\n    # via fastapi-cloud-cli\nsetproctitle==1.3.8\n    # via vllm\nsetuptools==79.0.1\n    # via\n    #   torch\n    #   triton\n    #   vllm\nshellingham==1.5.4\n    # via typer\nsix==1.17.0\n    # via vllm\nsniffio==1.3.1\n    # via openai\nsoundfile==0.14.0\n    # via mistral-common\nsoxr==1.1.0\n    # via mistral-common\nstarlette==1.10.0\n    # via\n    #   fastapi\n    #   prometheus-fastapi-instrumentator\nsympy==1.14.0\n    # via torch\ntiktoken==0.14.0\n    # via\n    #   mistral-common\n    #   vllm\ntokenizers==0.22.2\n    # via\n    #   transformers\n    #   vllm\ntorch==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   compressed-tensors\n    #   cut-cross-entropy\n    #   peft\n    #   torchaudio\n    #   torchvision\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\ntorchao==0.13.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth-zoo\ntorchaudio==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   vllm\ntorchvision==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\ntqdm==4.70.1\n    # via\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   peft\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\ntransformers==4.56.2\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   compressed-tensors\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntriton==3.4.0\n    # via\n    #   cut-cross-entropy\n    #   torch\n    #   unsloth\n    #   unsloth-zoo\n    #   xgrammar\ntrl==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via tyro\ntyper==0.27.3\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\ntyping-extensions==4.16.0\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   anyio\n    #   fastapi\n    #   httpx2\n    #   huggingface-hub\n    #   openai\n    #   opentelemetry-api\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   referencing\n    #   rich-toolkit\n    #   soundfile\n    #   starlette\n    #   torch\n    #   typeguard\n    #   typing-inspection\n    #   tyro\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntyping-inspection==0.4.4\n    # via\n    #   fastapi\n    #   pydantic\n    #   pydantic-settings\ntyro==1.0.16\n    # via\n    #   unsloth\n    #   unsloth-zoo\ntzdata==2026.6\n    # via python-dateutil\nunsloth==2025.10.9\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nunsloth-zoo==2025.10.10\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nurllib3==2.8.0\n    # via\n    #   requests\n    #   sentry-sdk\nuvicorn==0.54.0\n    # via\n    #   fastapi\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nuvloop==0.23.0\n    # via uvicorn\nvllm==0.10.2\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nwatchfiles==1.3.0\n    # via\n    #   uvicorn\n    #   vllm\nwebsockets==17.4\n    # via uvicorn\nwheel==0.48.0\n    # via\n    #   unsloth\n    #   unsloth-zoo\nwrapt==1.17.3\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nxformers==0.0.32.post1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\nxgrammar==0.1.23\n    # via vllm\nxxhash==4.0.2\n    # via datasets\nyarl==1.25.1\n    # via aiohttp\nzipp==4.1.2\n    # via importlib-metadata\n', 'split.txt': 'train_0001\ntrain_0004\ntrain_0015\ntrain_0016\ntrain_0017\ntrain_0031\ntrain_0063\ntrain_0066\ntrain_0075\ntrain_0080\ntrain_0088\ntrain_0090\ntrain_0095\ntrain_0106\ntrain_0110\ntrain_0117\ntrain_0130\ntrain_0133\ntrain_0141\ntrain_0144\ntrain_0145\ntrain_0147\ntrain_0162\ntrain_0163\ntrain_0164\ntrain_0169\ntrain_0172\ntrain_0181\ntrain_0194\ntrain_0196\ntrain_0203\ntrain_0207\ntrain_0214\ntrain_0216\ntrain_0219\ntrain_0225\ntrain_0228\ntrain_0231\ntrain_0233\ntrain_0236\ntrain_0240\ntrain_0255\ntrain_0278\ntrain_0285\ntrain_0304\ntrain_0313\ntrain_0316\ntrain_0321\ntrain_0330\ntrain_0344\ntrain_0345\ntrain_0346\ntrain_0350\ntrain_0354\ntrain_0357\ntrain_0378\ntrain_0381\ntrain_0395\ntrain_0405\ntrain_0422\ntrain_0435\ntrain_0442\ntrain_0453\ntrain_0457\ntrain_0468\ntrain_0469\ntrain_0476\ntrain_0480\ntrain_0481\ntrain_0486\ntrain_0497\ntrain_0500\ntrain_0505\ntrain_0508\ntrain_0518\ntrain_0536\ntrain_0538\ntrain_0550\ntrain_0554\ntrain_0560\ntrain_0563\ntrain_0572\ntrain_0577\ntrain_0596\ntrain_0598\ntrain_0618\ntrain_0621\ntrain_0624\ntrain_0625\ntrain_0628\ntrain_0664\ntrain_0689\ntrain_0694\ntrain_0719\ntrain_0729\ntrain_0752\ntrain_0764\ntrain_0770\ntrain_0772\ntrain_0777\ntrain_0779\ntrain_0787\ntrain_0790\ntrain_0800\ntrain_0805\ntrain_0814\ntrain_0815\ntrain_0819\ntrain_0820\ntrain_0834\ntrain_0857\ntrain_0864\ntrain_0867\ntrain_0870\ntrain_0875\ntrain_0923\ntrain_0933\ntrain_0935\ntrain_0943\ntrain_0945\ntrain_0947\ntrain_0948\ntrain_0953\ntrain_0958\ntrain_0959\ntrain_0960\ntrain_0977\ntrain_0978\ntrain_0991\ntrain_1000\ntrain_1014\ntrain_1016\ntrain_1019\ntrain_1021\ntrain_1049\ntrain_1060\ntrain_1062\ntrain_1070\ntrain_1081\ntrain_1089\ntrain_1091\ntrain_1096\ntrain_1097\ntrain_1109\ntrain_1117\ntrain_1118\ntrain_1123\ntrain_1124\ntrain_1128\ntrain_1137\ntrain_1147\ntrain_1148\ntrain_1155\ntrain_1160\ntrain_1163\ntrain_1175\ntrain_1180\ntrain_1188\ntrain_1201\ntrain_1206\ntrain_1209\ntrain_1215\ntrain_1216\ntrain_1233\ntrain_1234\ntrain_1246\ntrain_1248\ntrain_1249\ntrain_1253\ntrain_1265\ntrain_1278\ntrain_1289\ntrain_1295\ntrain_1300\ntrain_1305\ntrain_1310\ntrain_1311\ntrain_1317\ntrain_1325\ntrain_1346\ntrain_1351\ntrain_1352\ntrain_1353\ntrain_1355\ntrain_1356\ntrain_1363\ntrain_1373\ntrain_1379\ntrain_1380\ntrain_1384\ntrain_1399\ntrain_1405\ntrain_1408\ntrain_1432\ntrain_1433\ntrain_1437\ntrain_1440\ntrain_1448\ntrain_1458\ntrain_1460\ntrain_1471\ntrain_1472\ntrain_1474\ntrain_1475\ntrain_1476\ntrain_1478\ntrain_1486\ntrain_1491\ntrain_1497\ntrain_1506\ntrain_1512\ntrain_1519\ntrain_1526\ntrain_1528\ntrain_1533\ntrain_1540\ntrain_1542\ntrain_1544\ntrain_1546\ntrain_1556\ntrain_1557\ntrain_1560\ntrain_1571\ntrain_1572\ntrain_1589\ntrain_1598\ntrain_1606\ntrain_1608\ntrain_1615\ntrain_1617\ntrain_1623\ntrain_1653\ntrain_1654\ntrain_1658\ntrain_1661\ntrain_1668\ntrain_1697\ntrain_1702\ntrain_1703\ntrain_1705\ntrain_1707\ntrain_1712\ntrain_1713\ntrain_1715\ntrain_1746\n', 'split_manifest.json': '{\n  "seed": "english-split-20261104",\n  "train": 1300,\n  "validation": 245,\n  "original_valid_retained": 100,\n  "additional_valid": 145,\n  "strata": "first-tool app, capped call count, conditional keyword; proportional largest remainder",\n  "split_sha256": "ce713fbd99bf0bad81420935477f545af1b97fa96e5b606c6ea9377e465b7427",\n  "note": "ID-disjoint split. Semantic paraphrase groups have not been manually audited; old 100 are a development set.",\n  "requires_new_training": true,\n  "training_mode": "Independent local Planner/Caller from base"\n}\n', 'train_ids.txt': 'train_0002\ntrain_0003\ntrain_0005\ntrain_0006\ntrain_0007\ntrain_0008\ntrain_0009\ntrain_0011\ntrain_0012\ntrain_0013\ntrain_0014\ntrain_0018\ntrain_0019\ntrain_0020\ntrain_0025\ntrain_0026\ntrain_0027\ntrain_0028\ntrain_0029\ntrain_0030\ntrain_0032\ntrain_0033\ntrain_0034\ntrain_0035\ntrain_0036\ntrain_0037\ntrain_0038\ntrain_0039\ntrain_0040\ntrain_0041\ntrain_0042\ntrain_0043\ntrain_0045\ntrain_0046\ntrain_0047\ntrain_0048\ntrain_0049\ntrain_0050\ntrain_0052\ntrain_0055\ntrain_0057\ntrain_0059\ntrain_0060\ntrain_0061\ntrain_0062\ntrain_0064\ntrain_0065\ntrain_0068\ntrain_0069\ntrain_0070\ntrain_0071\ntrain_0072\ntrain_0074\ntrain_0076\ntrain_0078\ntrain_0079\ntrain_0081\ntrain_0082\ntrain_0083\ntrain_0084\ntrain_0085\ntrain_0086\ntrain_0087\ntrain_0089\ntrain_0091\ntrain_0092\ntrain_0093\ntrain_0094\ntrain_0096\ntrain_0097\ntrain_0098\ntrain_0100\ntrain_0103\ntrain_0104\ntrain_0105\ntrain_0107\ntrain_0108\ntrain_0109\ntrain_0111\ntrain_0112\ntrain_0113\ntrain_0115\ntrain_0116\ntrain_0119\ntrain_0121\ntrain_0122\ntrain_0123\ntrain_0124\ntrain_0125\ntrain_0126\ntrain_0127\ntrain_0128\ntrain_0129\ntrain_0131\ntrain_0132\ntrain_0134\ntrain_0135\ntrain_0136\ntrain_0137\ntrain_0138\ntrain_0139\ntrain_0140\ntrain_0142\ntrain_0143\ntrain_0146\ntrain_0148\ntrain_0149\ntrain_0150\ntrain_0151\ntrain_0152\ntrain_0153\ntrain_0154\ntrain_0155\ntrain_0156\ntrain_0157\ntrain_0158\ntrain_0159\ntrain_0161\ntrain_0165\ntrain_0166\ntrain_0168\ntrain_0170\ntrain_0171\ntrain_0173\ntrain_0174\ntrain_0176\ntrain_0177\ntrain_0178\ntrain_0179\ntrain_0180\ntrain_0182\ntrain_0183\ntrain_0184\ntrain_0185\ntrain_0186\ntrain_0187\ntrain_0188\ntrain_0189\ntrain_0190\ntrain_0191\ntrain_0192\ntrain_0193\ntrain_0195\ntrain_0197\ntrain_0199\ntrain_0200\ntrain_0201\ntrain_0202\ntrain_0204\ntrain_0205\ntrain_0206\ntrain_0208\ntrain_0209\ntrain_0210\ntrain_0211\ntrain_0212\ntrain_0213\ntrain_0215\ntrain_0217\ntrain_0220\ntrain_0221\ntrain_0222\ntrain_0224\ntrain_0226\ntrain_0227\ntrain_0229\ntrain_0230\ntrain_0232\ntrain_0234\ntrain_0235\ntrain_0237\ntrain_0238\ntrain_0239\ntrain_0241\ntrain_0242\ntrain_0243\ntrain_0244\ntrain_0245\ntrain_0246\ntrain_0247\ntrain_0248\ntrain_0249\ntrain_0250\ntrain_0251\ntrain_0252\ntrain_0253\ntrain_0254\ntrain_0256\ntrain_0257\ntrain_0258\ntrain_0259\ntrain_0260\ntrain_0261\ntrain_0262\ntrain_0263\ntrain_0264\ntrain_0265\ntrain_0266\ntrain_0267\ntrain_0268\ntrain_0269\ntrain_0270\ntrain_0271\ntrain_0272\ntrain_0273\ntrain_0274\ntrain_0275\ntrain_0276\ntrain_0277\ntrain_0279\ntrain_0280\ntrain_0281\ntrain_0282\ntrain_0283\ntrain_0284\ntrain_0286\ntrain_0287\ntrain_0288\ntrain_0289\ntrain_0290\ntrain_0291\ntrain_0292\ntrain_0293\ntrain_0294\ntrain_0295\ntrain_0296\ntrain_0297\ntrain_0298\ntrain_0299\ntrain_0300\ntrain_0301\ntrain_0302\ntrain_0303\ntrain_0306\ntrain_0307\ntrain_0308\ntrain_0309\ntrain_0310\ntrain_0311\ntrain_0312\ntrain_0314\ntrain_0315\ntrain_0317\ntrain_0318\ntrain_0319\ntrain_0320\ntrain_0322\ntrain_0323\ntrain_0324\ntrain_0325\ntrain_0326\ntrain_0327\ntrain_0328\ntrain_0329\ntrain_0331\ntrain_0332\ntrain_0333\ntrain_0334\ntrain_0335\ntrain_0336\ntrain_0337\ntrain_0338\ntrain_0339\ntrain_0340\ntrain_0341\ntrain_0342\ntrain_0343\ntrain_0347\ntrain_0348\ntrain_0349\ntrain_0351\ntrain_0352\ntrain_0353\ntrain_0355\ntrain_0356\ntrain_0358\ntrain_0359\ntrain_0360\ntrain_0361\ntrain_0362\ntrain_0363\ntrain_0364\ntrain_0365\ntrain_0366\ntrain_0367\ntrain_0368\ntrain_0369\ntrain_0370\ntrain_0371\ntrain_0372\ntrain_0373\ntrain_0374\ntrain_0375\ntrain_0376\ntrain_0377\ntrain_0379\ntrain_0380\ntrain_0382\ntrain_0383\ntrain_0384\ntrain_0385\ntrain_0386\ntrain_0387\ntrain_0388\ntrain_0389\ntrain_0390\ntrain_0391\ntrain_0392\ntrain_0393\ntrain_0394\ntrain_0396\ntrain_0397\ntrain_0398\ntrain_0399\ntrain_0400\ntrain_0401\ntrain_0402\ntrain_0403\ntrain_0404\ntrain_0406\ntrain_0407\ntrain_0408\ntrain_0409\ntrain_0410\ntrain_0411\ntrain_0412\ntrain_0413\ntrain_0414\ntrain_0415\ntrain_0416\ntrain_0417\ntrain_0418\ntrain_0419\ntrain_0420\ntrain_0421\ntrain_0423\ntrain_0424\ntrain_0425\ntrain_0426\ntrain_0427\ntrain_0428\ntrain_0429\ntrain_0430\ntrain_0431\ntrain_0432\ntrain_0433\ntrain_0434\ntrain_0436\ntrain_0437\ntrain_0439\ntrain_0440\ntrain_0441\ntrain_0443\ntrain_0444\ntrain_0445\ntrain_0446\ntrain_0447\ntrain_0448\ntrain_0449\ntrain_0450\ntrain_0451\ntrain_0452\ntrain_0454\ntrain_0455\ntrain_0456\ntrain_0458\ntrain_0459\ntrain_0460\ntrain_0461\ntrain_0462\ntrain_0463\ntrain_0464\ntrain_0465\ntrain_0466\ntrain_0467\ntrain_0470\ntrain_0471\ntrain_0472\ntrain_0473\ntrain_0474\ntrain_0475\ntrain_0477\ntrain_0478\ntrain_0479\ntrain_0482\ntrain_0483\ntrain_0484\ntrain_0485\ntrain_0488\ntrain_0490\ntrain_0491\ntrain_0493\ntrain_0494\ntrain_0495\ntrain_0496\ntrain_0498\ntrain_0499\ntrain_0501\ntrain_0502\ntrain_0503\ntrain_0504\ntrain_0506\ntrain_0507\ntrain_0509\ntrain_0510\ntrain_0511\ntrain_0512\ntrain_0513\ntrain_0514\ntrain_0515\ntrain_0516\ntrain_0517\ntrain_0520\ntrain_0523\ntrain_0526\ntrain_0527\ntrain_0528\ntrain_0529\ntrain_0530\ntrain_0531\ntrain_0532\ntrain_0533\ntrain_0534\ntrain_0535\ntrain_0537\ntrain_0539\ntrain_0540\ntrain_0541\ntrain_0543\ntrain_0544\ntrain_0545\ntrain_0546\ntrain_0548\ntrain_0551\ntrain_0556\ntrain_0557\ntrain_0558\ntrain_0559\ntrain_0562\ntrain_0565\ntrain_0567\ntrain_0568\ntrain_0570\ntrain_0571\ntrain_0573\ntrain_0574\ntrain_0575\ntrain_0578\ntrain_0579\ntrain_0581\ntrain_0583\ntrain_0584\ntrain_0587\ntrain_0588\ntrain_0589\ntrain_0593\ntrain_0594\ntrain_0595\ntrain_0597\ntrain_0599\ntrain_0600\ntrain_0601\ntrain_0602\ntrain_0603\ntrain_0604\ntrain_0605\ntrain_0606\ntrain_0607\ntrain_0608\ntrain_0609\ntrain_0610\ntrain_0611\ntrain_0612\ntrain_0613\ntrain_0614\ntrain_0615\ntrain_0616\ntrain_0617\ntrain_0619\ntrain_0620\ntrain_0622\ntrain_0623\ntrain_0626\ntrain_0627\ntrain_0629\ntrain_0630\ntrain_0631\ntrain_0632\ntrain_0633\ntrain_0634\ntrain_0635\ntrain_0636\ntrain_0637\ntrain_0638\ntrain_0639\ntrain_0640\ntrain_0641\ntrain_0642\ntrain_0643\ntrain_0645\ntrain_0647\ntrain_0648\ntrain_0651\ntrain_0652\ntrain_0653\ntrain_0654\ntrain_0655\ntrain_0657\ntrain_0660\ntrain_0665\ntrain_0666\ntrain_0668\ntrain_0669\ntrain_0671\ntrain_0672\ntrain_0673\ntrain_0674\ntrain_0675\ntrain_0676\ntrain_0677\ntrain_0678\ntrain_0679\ntrain_0680\ntrain_0681\ntrain_0682\ntrain_0683\ntrain_0684\ntrain_0685\ntrain_0686\ntrain_0687\ntrain_0688\ntrain_0690\ntrain_0691\ntrain_0692\ntrain_0693\ntrain_0695\ntrain_0696\ntrain_0697\ntrain_0698\ntrain_0699\ntrain_0700\ntrain_0701\ntrain_0702\ntrain_0703\ntrain_0704\ntrain_0705\ntrain_0706\ntrain_0707\ntrain_0708\ntrain_0709\ntrain_0710\ntrain_0711\ntrain_0712\ntrain_0713\ntrain_0714\ntrain_0715\ntrain_0716\ntrain_0717\ntrain_0720\ntrain_0721\ntrain_0724\ntrain_0725\ntrain_0728\ntrain_0730\ntrain_0731\ntrain_0732\ntrain_0734\ntrain_0735\ntrain_0736\ntrain_0738\ntrain_0739\ntrain_0740\ntrain_0741\ntrain_0743\ntrain_0744\ntrain_0745\ntrain_0746\ntrain_0747\ntrain_0748\ntrain_0749\ntrain_0751\ntrain_0753\ntrain_0755\ntrain_0757\ntrain_0760\ntrain_0761\ntrain_0765\ntrain_0766\ntrain_0767\ntrain_0769\ntrain_0771\ntrain_0773\ntrain_0774\ntrain_0775\ntrain_0776\ntrain_0778\ntrain_0780\ntrain_0781\ntrain_0782\ntrain_0783\ntrain_0784\ntrain_0785\ntrain_0786\ntrain_0788\ntrain_0789\ntrain_0791\ntrain_0792\ntrain_0793\ntrain_0794\ntrain_0795\ntrain_0796\ntrain_0797\ntrain_0798\ntrain_0799\ntrain_0801\ntrain_0802\ntrain_0803\ntrain_0804\ntrain_0806\ntrain_0807\ntrain_0808\ntrain_0809\ntrain_0810\ntrain_0811\ntrain_0812\ntrain_0813\ntrain_0816\ntrain_0817\ntrain_0821\ntrain_0822\ntrain_0823\ntrain_0824\ntrain_0825\ntrain_0826\ntrain_0827\ntrain_0828\ntrain_0829\ntrain_0830\ntrain_0831\ntrain_0832\ntrain_0833\ntrain_0835\ntrain_0836\ntrain_0837\ntrain_0838\ntrain_0839\ntrain_0840\ntrain_0841\ntrain_0842\ntrain_0843\ntrain_0846\ntrain_0847\ntrain_0848\ntrain_0849\ntrain_0850\ntrain_0851\ntrain_0852\ntrain_0853\ntrain_0854\ntrain_0855\ntrain_0856\ntrain_0858\ntrain_0859\ntrain_0860\ntrain_0861\ntrain_0862\ntrain_0863\ntrain_0865\ntrain_0866\ntrain_0868\ntrain_0869\ntrain_0871\ntrain_0872\ntrain_0873\ntrain_0874\ntrain_0876\ntrain_0877\ntrain_0878\ntrain_0879\ntrain_0880\ntrain_0881\ntrain_0882\ntrain_0883\ntrain_0884\ntrain_0885\ntrain_0886\ntrain_0887\ntrain_0888\ntrain_0889\ntrain_0890\ntrain_0891\ntrain_0892\ntrain_0893\ntrain_0894\ntrain_0895\ntrain_0896\ntrain_0897\ntrain_0898\ntrain_0899\ntrain_0900\ntrain_0901\ntrain_0902\ntrain_0903\ntrain_0904\ntrain_0905\ntrain_0906\ntrain_0907\ntrain_0908\ntrain_0909\ntrain_0910\ntrain_0911\ntrain_0912\ntrain_0913\ntrain_0915\ntrain_0919\ntrain_0920\ntrain_0921\ntrain_0922\ntrain_0925\ntrain_0926\ntrain_0929\ntrain_0930\ntrain_0931\ntrain_0932\ntrain_0934\ntrain_0936\ntrain_0937\ntrain_0938\ntrain_0939\ntrain_0940\ntrain_0942\ntrain_0944\ntrain_0946\ntrain_0949\ntrain_0950\ntrain_0951\ntrain_0952\ntrain_0955\ntrain_0956\ntrain_0957\ntrain_0961\ntrain_0962\ntrain_0963\ntrain_0964\ntrain_0965\ntrain_0966\ntrain_0967\ntrain_0968\ntrain_0969\ntrain_0970\ntrain_0971\ntrain_0972\ntrain_0973\ntrain_0974\ntrain_0975\ntrain_0976\ntrain_0979\ntrain_0980\ntrain_0981\ntrain_0982\ntrain_0983\ntrain_0984\ntrain_0986\ntrain_0988\ntrain_0990\ntrain_0992\ntrain_0993\ntrain_0994\ntrain_0995\ntrain_0996\ntrain_0997\ntrain_0998\ntrain_0999\ntrain_1002\ntrain_1003\ntrain_1004\ntrain_1005\ntrain_1006\ntrain_1009\ntrain_1010\ntrain_1011\ntrain_1012\ntrain_1013\ntrain_1015\ntrain_1017\ntrain_1020\ntrain_1022\ntrain_1023\ntrain_1024\ntrain_1025\ntrain_1026\ntrain_1027\ntrain_1028\ntrain_1029\ntrain_1030\ntrain_1031\ntrain_1032\ntrain_1033\ntrain_1034\ntrain_1035\ntrain_1036\ntrain_1038\ntrain_1039\ntrain_1040\ntrain_1041\ntrain_1044\ntrain_1046\ntrain_1050\ntrain_1052\ntrain_1053\ntrain_1054\ntrain_1055\ntrain_1056\ntrain_1057\ntrain_1058\ntrain_1059\ntrain_1061\ntrain_1063\ntrain_1064\ntrain_1066\ntrain_1067\ntrain_1068\ntrain_1071\ntrain_1072\ntrain_1074\ntrain_1075\ntrain_1076\ntrain_1077\ntrain_1078\ntrain_1079\ntrain_1080\ntrain_1082\ntrain_1083\ntrain_1084\ntrain_1085\ntrain_1086\ntrain_1087\ntrain_1088\ntrain_1090\ntrain_1093\ntrain_1094\ntrain_1095\ntrain_1098\ntrain_1099\ntrain_1100\ntrain_1103\ntrain_1104\ntrain_1105\ntrain_1106\ntrain_1107\ntrain_1110\ntrain_1111\ntrain_1113\ntrain_1114\ntrain_1115\ntrain_1116\ntrain_1119\ntrain_1121\ntrain_1122\ntrain_1125\ntrain_1127\ntrain_1130\ntrain_1131\ntrain_1132\ntrain_1133\ntrain_1134\ntrain_1135\ntrain_1136\ntrain_1138\ntrain_1139\ntrain_1140\ntrain_1141\ntrain_1142\ntrain_1143\ntrain_1144\ntrain_1145\ntrain_1146\ntrain_1149\ntrain_1150\ntrain_1151\ntrain_1153\ntrain_1154\ntrain_1156\ntrain_1157\ntrain_1158\ntrain_1159\ntrain_1161\ntrain_1162\ntrain_1164\ntrain_1165\ntrain_1166\ntrain_1167\ntrain_1168\ntrain_1169\ntrain_1171\ntrain_1172\ntrain_1173\ntrain_1174\ntrain_1176\ntrain_1177\ntrain_1178\ntrain_1179\ntrain_1181\ntrain_1183\ntrain_1184\ntrain_1185\ntrain_1186\ntrain_1187\ntrain_1191\ntrain_1192\ntrain_1193\ntrain_1194\ntrain_1195\ntrain_1196\ntrain_1197\ntrain_1199\ntrain_1200\ntrain_1202\ntrain_1203\ntrain_1204\ntrain_1205\ntrain_1207\ntrain_1208\ntrain_1210\ntrain_1211\ntrain_1212\ntrain_1213\ntrain_1214\ntrain_1217\ntrain_1218\ntrain_1219\ntrain_1221\ntrain_1222\ntrain_1223\ntrain_1224\ntrain_1226\ntrain_1227\ntrain_1228\ntrain_1229\ntrain_1231\ntrain_1236\ntrain_1237\ntrain_1238\ntrain_1240\ntrain_1241\ntrain_1242\ntrain_1244\ntrain_1245\ntrain_1247\ntrain_1250\ntrain_1252\ntrain_1254\ntrain_1255\ntrain_1256\ntrain_1257\ntrain_1259\ntrain_1260\ntrain_1261\ntrain_1262\ntrain_1263\ntrain_1264\ntrain_1266\ntrain_1268\ntrain_1270\ntrain_1271\ntrain_1274\ntrain_1275\ntrain_1276\ntrain_1277\ntrain_1279\ntrain_1280\ntrain_1281\ntrain_1282\ntrain_1283\ntrain_1284\ntrain_1285\ntrain_1286\ntrain_1287\ntrain_1290\ntrain_1291\ntrain_1292\ntrain_1293\ntrain_1294\ntrain_1296\ntrain_1297\ntrain_1299\ntrain_1301\ntrain_1302\ntrain_1303\ntrain_1306\ntrain_1308\ntrain_1312\ntrain_1313\ntrain_1314\ntrain_1315\ntrain_1316\ntrain_1318\ntrain_1319\ntrain_1320\ntrain_1322\ntrain_1323\ntrain_1324\ntrain_1326\ntrain_1327\ntrain_1328\ntrain_1329\ntrain_1330\ntrain_1331\ntrain_1332\ntrain_1333\ntrain_1334\ntrain_1335\ntrain_1336\ntrain_1337\ntrain_1338\ntrain_1339\ntrain_1340\ntrain_1341\ntrain_1343\ntrain_1344\ntrain_1345\ntrain_1347\ntrain_1348\ntrain_1349\ntrain_1354\ntrain_1357\ntrain_1358\ntrain_1359\ntrain_1360\ntrain_1361\ntrain_1362\ntrain_1364\ntrain_1365\ntrain_1366\ntrain_1367\ntrain_1368\ntrain_1369\ntrain_1370\ntrain_1371\ntrain_1372\ntrain_1374\ntrain_1375\ntrain_1376\ntrain_1377\ntrain_1378\ntrain_1381\ntrain_1382\ntrain_1385\ntrain_1386\ntrain_1387\ntrain_1388\ntrain_1389\ntrain_1390\ntrain_1391\ntrain_1392\ntrain_1394\ntrain_1395\ntrain_1396\ntrain_1397\ntrain_1398\ntrain_1400\ntrain_1401\ntrain_1402\ntrain_1403\ntrain_1404\ntrain_1406\ntrain_1407\ntrain_1409\ntrain_1411\ntrain_1412\ntrain_1413\ntrain_1414\ntrain_1415\ntrain_1417\ntrain_1418\ntrain_1419\ntrain_1422\ntrain_1423\ntrain_1424\ntrain_1425\ntrain_1426\ntrain_1427\ntrain_1428\ntrain_1429\ntrain_1430\ntrain_1431\ntrain_1434\ntrain_1435\ntrain_1436\ntrain_1438\ntrain_1439\ntrain_1441\ntrain_1442\ntrain_1445\ntrain_1446\ntrain_1447\ntrain_1449\ntrain_1450\ntrain_1451\ntrain_1452\ntrain_1453\ntrain_1454\ntrain_1457\ntrain_1462\ntrain_1463\ntrain_1464\ntrain_1469\ntrain_1477\ntrain_1479\ntrain_1480\ntrain_1481\ntrain_1482\ntrain_1483\ntrain_1484\ntrain_1485\ntrain_1487\ntrain_1489\ntrain_1490\ntrain_1493\ntrain_1494\ntrain_1495\ntrain_1496\ntrain_1498\ntrain_1499\ntrain_1500\ntrain_1502\ntrain_1503\ntrain_1504\ntrain_1505\ntrain_1507\ntrain_1508\ntrain_1509\ntrain_1510\ntrain_1511\ntrain_1513\ntrain_1514\ntrain_1515\ntrain_1516\ntrain_1517\ntrain_1518\ntrain_1520\ntrain_1521\ntrain_1522\ntrain_1523\ntrain_1524\ntrain_1527\ntrain_1529\ntrain_1531\ntrain_1534\ntrain_1537\ntrain_1538\ntrain_1541\ntrain_1543\ntrain_1545\ntrain_1547\ntrain_1548\ntrain_1549\ntrain_1550\ntrain_1551\ntrain_1552\ntrain_1553\ntrain_1555\ntrain_1558\ntrain_1559\ntrain_1561\ntrain_1562\ntrain_1563\ntrain_1564\ntrain_1565\ntrain_1566\ntrain_1567\ntrain_1568\ntrain_1569\ntrain_1570\ntrain_1573\ntrain_1575\ntrain_1576\ntrain_1577\ntrain_1578\ntrain_1579\ntrain_1580\ntrain_1581\ntrain_1582\ntrain_1583\ntrain_1584\ntrain_1585\ntrain_1587\ntrain_1588\ntrain_1590\ntrain_1591\ntrain_1592\ntrain_1593\ntrain_1594\ntrain_1595\ntrain_1597\ntrain_1599\ntrain_1600\ntrain_1602\ntrain_1603\ntrain_1604\ntrain_1605\ntrain_1607\ntrain_1609\ntrain_1610\ntrain_1611\ntrain_1612\ntrain_1616\ntrain_1618\ntrain_1620\ntrain_1621\ntrain_1622\ntrain_1624\ntrain_1626\ntrain_1628\ntrain_1629\ntrain_1630\ntrain_1631\ntrain_1632\ntrain_1633\ntrain_1634\ntrain_1635\ntrain_1636\ntrain_1638\ntrain_1641\ntrain_1642\ntrain_1643\ntrain_1644\ntrain_1645\ntrain_1646\ntrain_1647\ntrain_1648\ntrain_1649\ntrain_1650\ntrain_1651\ntrain_1652\ntrain_1655\ntrain_1657\ntrain_1659\ntrain_1660\ntrain_1662\ntrain_1663\ntrain_1664\ntrain_1665\ntrain_1667\ntrain_1669\ntrain_1670\ntrain_1672\ntrain_1673\ntrain_1674\ntrain_1675\ntrain_1676\ntrain_1677\ntrain_1678\ntrain_1679\ntrain_1681\ntrain_1682\ntrain_1683\ntrain_1685\ntrain_1686\ntrain_1687\ntrain_1688\ntrain_1689\ntrain_1691\ntrain_1692\ntrain_1694\ntrain_1695\ntrain_1696\ntrain_1698\ntrain_1700\ntrain_1701\ntrain_1704\ntrain_1706\ntrain_1709\ntrain_1711\ntrain_1714\ntrain_1716\ntrain_1717\ntrain_1720\ntrain_1721\ntrain_1722\ntrain_1723\ntrain_1725\ntrain_1728\ntrain_1729\ntrain_1730\ntrain_1733\ntrain_1734\ntrain_1735\ntrain_1736\ntrain_1738\ntrain_1739\ntrain_1740\ntrain_1741\ntrain_1742\ntrain_1743\ntrain_1748\ntrain_1749\ntrain_1750\ntrain_1751\ntrain_1753\n', 'umilife/completion_loss.py': '"""Memory-bounded causal CE on assistant tokens; Unsloth returns hidden states."""\nimport os\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\ndef completion_loss(model,batch,chunk_size=None):\n    chunk_size=chunk_size or int(os.environ.get("LIFEMATE_LOSS_CHUNK","512"))\n    assert chunk_size>0\n    # UNSLOTH_RETURN_HIDDEN_STATES=1 bypasses its fused vocabulary-sized loss allocation.\n    output=model(**batch,use_cache=False,return_dict=True)\n    hidden=output.logits\n    head=model.get_output_embeddings()\n    assert hidden.shape[-1]==head.weight.shape[1],\'Expected Unsloth hidden-state output, not vocabulary logits\'\n    labels=batch[\'labels\'][:,1:]\n    mask=labels!=-100\n    selected=hidden[:,:-1][mask];targets=labels[mask]\n    assert targets.numel()>0\n    def chunk_nll(h,t):\n        logits=F.linear(h.to(head.weight.dtype),head.weight,head.bias)\n        return F.cross_entropy(logits.float(),t,reduction=\'sum\')\n    total=hidden.new_zeros((),dtype=torch.float32)\n    for start in range(0,len(targets),chunk_size):\n        h=selected[start:start+chunk_size];t=targets[start:start+chunk_size]\n        if os.environ.get("LOSS_PROJECTION_CHECKPOINT","0")=="1" and torch.is_grad_enabled() and h.requires_grad:\n            loss=checkpoint(chunk_nll,h,t,use_reentrant=False)\n        else:loss=chunk_nll(h,t)\n        total=total+loss\n    return total/targets.numel()\n', 'umilife/ddp_engine.py': '"""Explicit, symmetric DDP collectives; no Trainer monkey-patched control flow."""\nimport contextlib,hashlib,json,math,random,shutil,time,os,signal\nfrom pathlib import Path\nimport torch\nimport numpy as np\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\n\n\ndef batches(lengths,epoch,rank,world,batch_size=4):\n    # Shuffle then length-bucket across both GPUs to reduce padding/stragglers.\n    indices=list(range(len(lengths)));random.Random(42+epoch).shuffle(indices)\n    width=batch_size*world\n    ordered=[]\n    for start in range(0,len(indices),width*32):\n        ordered.extend(sorted(indices[start:start+width*32],key=lambda i:lengths[i],reverse=True))\n    padding=(-len(ordered))%width\n    ordered.extend((ordered*math.ceil(padding/max(1,len(ordered))))[:padding])\n    return [ordered[i+rank*batch_size:i+(rank+1)*batch_size] for i in range(0,len(ordered),width)]\n\n\ndef state_digest(model):\n    h=hashlib.sha256()\n    for name,p in sorted(model.named_parameters()):\n        if p.requires_grad:\n            h.update(name.encode());h.update(p.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())\n    return h.hexdigest()\n\n\ndef agreement(value):\n    values=[None]*dist.get_world_size();dist.all_gather_object(values,value)\n    assert len(set(values))==1,values\n    return value\n\n\ndef weighted_eval(model,items,collate,device,amp,batch_size=4,loss_function=None):\n    # No DDP forward: rank workloads can differ by one validation example.\n    model.eval();rank=dist.get_rank();world=dist.get_world_size()\n    totals=torch.zeros(2,device=device,dtype=torch.float64)\n    local=sorted(range(rank,len(items),world),key=lambda i:len(items[i][\'input_ids\']))\n    with torch.inference_mode():\n        for start in range(0,len(local),batch_size):\n            x={k:v.to(device) for k,v in collate([items[i] for i in local[start:start+batch_size]]).items()}\n            n=(x[\'labels\'][:,1:]!=-100).sum()\n            with amp():loss=loss_function(model,x) if loss_function else model(**x,use_cache=False,return_dict=True).loss\n            totals[0]+=loss.double()*n;totals[1]+=n\n    dist.all_reduce(totals)\n    result=(totals[0]/totals[1]).item()\n    assert math.isfinite(result) and totals[1]>0,\'Nonfinite validation loss\'\n    model.train()\n    return result\n\n\nclass LossModule(torch.nn.Module):\n    # DDP static_graph must see the loss tensor directly even for custom ModelOutput objects.\n    def __init__(self,model,loss_function=None):super().__init__();self.model=model;self.loss_function=loss_function\n    def forward(self,**kwargs):\n        if self.loss_function:return self.loss_function(self.model,{k:v for k,v in kwargs.items() if k not in ("use_cache","return_dict")})\n        return self.model(**kwargs).loss\n\n\ndef fit(model,train,valid,collate,device,output,epochs,lr,save_model,load_model,\n        eval_steps=50,patience=3,min_delta=.001,amp_dtype=None,train_batch_size=4,eval_batch_size=4,loss_function=None,max_steps=None):\n    rank=dist.get_rank();world=dist.get_world_size();assert world==2\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    latest=output/\'latest.json\'\n    assert not (output/\'best\').exists() or latest.exists(),\'Incomplete legacy checkpoint cannot resume\'\n    params=[p for p in model.parameters() if p.requires_grad]\n    wrapped=DistributedDataParallel(LossModule(model,loss_function),device_ids=[device.index] if device.type==\'cuda\' else None,\n        broadcast_buffers=False,find_unused_parameters=False,static_graph=True)\n    amp=(lambda:torch.autocast(device_type=device.type,dtype=amp_dtype)) if amp_dtype else contextlib.nullcontext\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=(device.type==\'cuda\' and amp_dtype==torch.float16),init_scale=1024)\n    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=.01)\n    lengths=[len(x[\'input_ids\']) for x in train]\n    steps_per_epoch=len(batches(lengths,0,rank,world,train_batch_size));total=steps_per_epoch*epochs\n    if max_steps is not None:total=min(total,max_steps)\n    warmup=max(1,math.ceil(.03*total))\n    def factor(step):\n        if step<warmup:return (step+1)/warmup\n        return .5*(1+math.cos(math.pi*min(1.,(step-warmup)/max(1,total-warmup))))\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,factor)\n    initial=agreement(state_digest(model))\n    # Resume loads the saved baseline; throughput mode skips duplicate baseline evaluation.\n    before=None\n    if not latest.exists() and os.environ.get(\'INITIAL_EVAL\',\'0\')==\'1\':\n        if rank==0:print(json.dumps({\'event\':\'initial_validation_start\',\'records\':len(valid)}),flush=True)\n        before=weighted_eval(model,valid,collate,device,amp,eval_batch_size,loss_function)\n        if rank==0:print(json.dumps({\'event\':\'initial_validation_end\',\'eval_loss\':before}),flush=True)\n    elif rank==0:print(json.dumps({\'event\':\'initial_validation_skipped\',\'resume\':latest.exists()}),flush=True)\n    eval_seconds=0.;last_log=time.monotonic();last_log_step=0\n    best=math.inf;significant=math.inf;bad=0;best_step=0;history=[]\n    step=0;updates=0;skipped=0;stop=False;started=time.monotonic()\n    last_epoch=0.;saved_hash=None\n    resume_epoch=resume_batch=0\n    train_seconds=checkpoint_seconds=0.;interval_seconds=0.;interval_input=interval_target=interval_padded=0\n    interrupted=[False]\n    signal.signal(signal.SIGTERM,lambda *_:interrupted.__setitem__(0,True))\n    signature=os.environ.get(\'TRAIN_RESUME_SIGNATURE\',\'\')\n    def persist(epoch,next_batch):\n        nonlocal checkpoint_seconds\n        checkpoint_started=time.monotonic()\n        rng={\'numpy\':np.random.get_state(),\'python\':random.getstate(),\'cpu\':torch.get_rng_state(),\n             \'cuda\':torch.cuda.get_rng_state(device) if device.type==\'cuda\' else None}\n        rngs=[None]*world;dist.all_gather_object(rngs,rng)\n        dist.barrier()\n        if rank==0:\n            folder=output/f\'step-{step:08d}\';temp=output/\'saving.tmp\'\n            if temp.exists():shutil.rmtree(temp)\n            temp.mkdir();save_model(model,temp)\n            if (output/\'best\').exists():shutil.copytree(output/\'best\',temp/\'best_snapshot\')\n            torch.save({\'optimizer\':optimizer.state_dict(),\'scheduler\':scheduler.state_dict(),\n                \'scaler\':scaler.state_dict(),\'rng\':rngs,\'signature\':signature,\'world_size\':world,\n                \'epoch\':epoch,\'next_batch\':next_batch,\'step\':step,\'updates\':updates,\'skipped\':skipped,\n                \'best\':best,\'significant\':significant,\'bad\':bad,\'best_step\':best_step,\'history\':history,\n                \'saved_hash\':saved_hash,\'initial\':initial,\'before\':before,\'last_epoch\':last_epoch,\n                \'eval_seconds\':eval_seconds,\'total\':total},temp/\'training_state.pt\')\n            if folder.exists():shutil.rmtree(folder)\n            temp.rename(folder)\n            pointer=output/\'latest.tmp\';pointer.write_text(json.dumps({\'folder\':folder.name,\'step\':step}));pointer.replace(latest)\n            for old in sorted(output.glob(\'step-*\'))[:-2]:shutil.rmtree(old)\n        dist.barrier()\n        elapsed=time.monotonic()-checkpoint_started;checkpoint_seconds+=elapsed\n        if rank==0:print(json.dumps({\'event\':\'checkpoint_saved\',\'step\':step,\'seconds\':elapsed}),flush=True)\n    if latest.exists():\n        folder=output/json.loads(latest.read_text())[\'folder\']\n        state=torch.load(folder/\'training_state.pt\',map_location=\'cpu\',weights_only=False)\n        assert state[\'signature\']==signature and state[\'world_size\']==world and state[\'total\']==total,\'Resume settings mismatch\'\n        if rank==0:\n            if (output/\'best\').exists():shutil.rmtree(output/\'best\')\n            if (folder/\'best_snapshot\').exists():shutil.copytree(folder/\'best_snapshot\',output/\'best\')\n        dist.barrier()\n        load_model(model,folder);optimizer.load_state_dict(state[\'optimizer\']);scheduler.load_state_dict(state[\'scheduler\']);scaler.load_state_dict(state[\'scaler\'])\n        step=state[\'step\'];updates=state[\'updates\'];skipped=state[\'skipped\'];best=state[\'best\'];significant=state[\'significant\'];bad=state[\'bad\'];best_step=state[\'best_step\'];history=state[\'history\'];saved_hash=state[\'saved_hash\'];initial=state[\'initial\'];before=state[\'before\'];last_epoch=state[\'last_epoch\'];eval_seconds=state[\'eval_seconds\']\n        resume_epoch=state[\'epoch\'];resume_batch=state[\'next_batch\'];stop=bad>=patience\n        np.random.set_state(state[\'rng\'][rank][\'numpy\']);random.setstate(state[\'rng\'][rank][\'python\']);torch.set_rng_state(state[\'rng\'][rank][\'cpu\'])\n        if device.type==\'cuda\':torch.cuda.set_rng_state(state[\'rng\'][rank][\'cuda\'],device)\n        last_log_step=step\n        print(json.dumps({\'resumed_step\':step,\'stage_checkpoint\':str(folder)}),flush=True)\n    for epoch in range(epochs):\n        if epoch<resume_epoch:continue\n\n        local_batches=batches(lengths,epoch,rank,world,train_batch_size)\n        for batch_index,indices in enumerate(local_batches):\n            if epoch==resume_epoch and batch_index<resume_batch:continue\n            if step>=total or bad>=patience:break\n            step_started=time.monotonic()\n            model.train();optimizer.zero_grad(set_to_none=True)\n            x={k:v.to(device) for k,v in collate([train[i] for i in indices]).items()}\n            n=(x[\'labels\'][:,1:]!=-100).sum()\n            global_n=n.clone();dist.all_reduce(global_n)\n            with amp():\n                raw_loss=wrapped(**x,use_cache=False,return_dict=True)\n                # DDP averages rank gradients; recover true global token-mean gradient.\n                loss=raw_loss*(world*n/global_n)\n            finite=torch.isfinite(loss).to(torch.int32);dist.all_reduce(finite,op=dist.ReduceOp.MIN)\n            assert finite.item()==1,\'Nonfinite training loss on a rank\'\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            norm=torch.nn.utils.clip_grad_norm_(params,1.)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            overflow=scaler.get_scale()<old_scale\n            flag=torch.tensor(int(overflow),device=device);dist.all_reduce(flag)\n            assert flag.item() in (0,world),"AMP overflow decisions diverged"\n            if overflow:skipped+=1\n            else:scheduler.step();updates+=1\n            step+=1;last_epoch=epoch+(batch_index+1)/len(local_batches)\n            elapsed=time.monotonic()-step_started;train_seconds+=elapsed;interval_seconds+=elapsed\n            interval_input+=int(x[\'attention_mask\'].sum().item());interval_target+=int(n.item());interval_padded+=x[\'input_ids\'].numel()\n            if step%10==0 or step==1:\n                count=torch.tensor([interval_input,interval_target,interval_padded],device=device,dtype=torch.float64);dist.all_reduce(count)\n                duration=torch.tensor(interval_seconds,device=device,dtype=torch.float64);dist.all_reduce(duration,op=dist.ReduceOp.MAX)\n                report={\'step\':step,\'planned_steps\':total,\'epoch\':last_epoch,\'local_loss\':raw_loss.item(),\n                        \'lr\':optimizer.param_groups[0][\'lr\'],\'grad_norm\':float(norm),\'optimizer_updates\':updates,\n                        \'train_seconds_per_step\':duration.item()/(step-last_log_step),\n                        \'input_tokens_per_second\':count[0].item()/max(duration.item(),1e-9),\n                        \'assistant_tokens_per_second\':count[1].item()/max(duration.item(),1e-9),\n                        \'padding_fraction\':1-count[0].item()/max(count[2].item(),1),\n                        \'peak_allocated_gb\':torch.cuda.max_memory_allocated(device)/2**30 if device.type==\'cuda\' else 0,\n                        \'seconds_per_step_including_eval\':(time.monotonic()-last_log)/(step-last_log_step)}\n                last_log=time.monotonic();last_log_step=step\n                interval_input=interval_target=interval_padded=0;interval_seconds=0.\n                if rank==0:print(json.dumps(report),flush=True)\n            at_end=step==total\n            if step%eval_steps==0 or at_end:\n                if rank==0:print(json.dumps({\'event\':\'validation_start\',\'step\':step,\'records\':len(valid)}),flush=True)\n                eval_started=time.monotonic()\n                metric=weighted_eval(model,valid,collate,device,amp,eval_batch_size,loss_function)\n                duration=time.monotonic()-eval_started;eval_seconds+=duration\n                improved=metric<best\n                if metric<significant-min_delta:significant=metric;bad=0\n                else:bad+=1\n                # Every rank computes the same scalar/state; only I/O is rank-zero-only.\n                if improved:\n                    best=metric;best_step=step\n                    dist.barrier()\n                    if rank==0:\n                        temp=output/\'best.tmp\'\n                        if temp.exists():shutil.rmtree(temp)\n                        temp.mkdir();save_model(model,temp)\n                        (temp/\'checkpoint.json\').write_text(json.dumps({\'step\':step,\'eval_loss\':metric,\'epoch\':last_epoch}))\n                        if (output/\'best\').exists():shutil.rmtree(output/\'best\')\n                        temp.rename(output/\'best\')\n                    dist.barrier()\n                    saved_hash=agreement(state_digest(model))\n                event={\'step\':step,\'epoch\':last_epoch,\'eval_loss\':metric,\'best_step\':best_step,\n                       \'best_loss\':best,\'patience_used\':bad,\'improved\':improved,\'evaluation_seconds\':duration}\n                history.append(event)\n                if rank==0:print(json.dumps(event),flush=True)\n                stop=bad>=patience\n                agreement(json.dumps([step,best,best_step,bad,stop]))\n            pause=torch.tensor(int(interrupted[0] or step==int(os.environ.get(\'SMOKE_PAUSE_STEP\',\'-1\')) or time.time()>=float(os.environ.get(\'TRAIN_DEADLINE\',\'inf\'))),device=device)\n            dist.all_reduce(pause,op=dist.ReduceOp.MAX)\n            if step%int(os.environ.get(\'SAVE_STEPS\',\'25\'))==0 or step%eval_steps==0 or at_end or stop or pause.item():\n                persist(epoch,batch_index+1)\n            if pause.item():raise SystemExit(75)\n            if stop or at_end:break\n        if stop or step>=total:break\n    assert updates>0 and best_step>0\n    dist.barrier();load_model(model,output/\'best\');dist.barrier()\n    final=agreement(state_digest(model));assert final==saved_hash and final!=initial\n    after=best\n    if os.environ.get(\'VERIFY_RESTORED_EVAL\',\'0\')==\'1\':\n        after=weighted_eval(model,valid,collate,device,amp,eval_batch_size,loss_function)\n        assert abs(after-best)<max(.005,abs(best)*.005),\'Best checkpoint restoration changed loss\'\n    result={\'world_size\':world,\'micro_batch\':train_batch_size,\'eval_batch_per_device\':eval_batch_size,\'gradient_accumulation\':1,\n            \'effective_batch\':train_batch_size*world,\'epochs\':epochs,\'actual_epoch\':last_epoch,\'global_step\':step,\n            \'optimizer_updates\':updates,\'amp_skipped_updates\':skipped,\'planned_steps\':total,\n            \'learning_rate\':lr,\'scheduler\':\'cosine\',\'warmup_ratio\':.03,\n            \'early_stopping\':{\'patience\':patience,\'min_delta\':min_delta,\'eval_steps\':eval_steps,\n                \'restore_best\':True,\'best_step\':best_step,\'best_metric\':best,\'stopped_early\':stop},\n            \'before\':{\'eval_loss\':before},\'after\':{\'eval_loss\':after,\'source\':\'rerun\' if os.environ.get(\'VERIFY_RESTORED_EVAL\',\'0\')==\'1\' else \'saved_validation_of_hash_verified_best\'},\'history\':history,\n            \'initial_trainable_sha256\':initial,\'final_trainable_sha256\':final,\n            \'wall_s\':time.monotonic()-started,\'train_padding_examples_per_epoch\':(-len(train))%(train_batch_size*world),\n            \'training_seconds_this_session\':train_seconds,\'checkpoint_seconds_this_session\':checkpoint_seconds,\'validation_repeats\':0,\'scheduled_evaluation_seconds\':eval_seconds}\n    dist.barrier();del wrapped\n    return result\n', 'umilife/evidence_labels.py': '"""English evidence compiler authored by Codex; not an independent LLM teacher.\nCurrent labels identify the supervised action, but do not supply its future result.\nOnly values traceable to visible evidence are serialized as facts.\n"""\nimport json,re,unicodedata\nfrom datetime import datetime,timedelta\nVERSION=\'evidence-goals-v3\'\n\ndef plain(s):\n    return \'\'.join(c for c in unicodedata.normalize(\'NFD\',s.lower().replace(\'đ\',\'d\')) if unicodedata.category(c)!=\'Mn\')\n\ndef records(value):\n    if isinstance(value,dict):\n        yield value\n        for v in value.values():yield from records(v)\n    elif isinstance(value,list):\n        for v in value:yield from records(v)\n\ndef scalars(value):\n    if isinstance(value,dict):\n        for v in value.values():yield from scalars(v)\n    elif isinstance(value,list):\n        for v in value:yield from scalars(v)\n    else:yield value\n\ndef visible(row,trajectory):\n    return [row.get(\'context\',{}),row.get(\'history\',[]),[p[\'observation\'] for p in trajectory]]\n\ndef supported(value,row,trajectory):\n    return (isinstance(value,str) and value in row[\'audio_text\']) or any(type(v) is type(value) and v==value for source in visible(row,trajectory) for v in scalars(source))\n\ndef number(text):\n    text=plain(text).replace(\',\',\'.\').strip()\n    if re.fullmatch(r\'\\d+(?:\\.\\d+)?\',text):return float(text)\n    small={\'khong\':0,\'mot\':1,\'hai\':2,\'ba\':3,\'bon\':4,\'tu\':4,\'nam\':5,\'lam\':5,\'sau\':6,\'bay\':7,\'tam\':8,\'chin\':9,\'muoi\':10}\n    total=chunk=current=0\n    for token in text.split():\n        if re.fullmatch(r\'\\d+(?:\\.\\d+)?\',token):current=float(token)\n        elif token in small:\n            if token==\'muoi\':current=max(1,current)*10\n            else:current+=small[token]\n        elif token==\'tram\':current=max(1,current)*100\n        elif token in (\'nghin\',\'ngan\',\'trieu\',\'ty\'):\n            scale={\'nghin\':1000,\'ngan\':1000,\'trieu\':10**6,\'ty\':10**9}[token]\n            chunk+=current;total+=chunk*scale;chunk=current=0\n        elif token in (\'le\',\'linh\'):continue\n        else:return None\n    return total+chunk+current\n\ndef constraints(row):\n    text=plain(row[\'audio_text\']);out=[];max_amount=None;days=None\n    m=re.search(r\'(?:trong|toi han trong)\\s+(\\d+|mot|hai|ba|bon|nam|sau|bay|tam|chin|muoi)\\s+ngay\',text)\n    if m:days=int(number(m[1]));out.append(f\'Use the next {days} days as the due-date window.\')\n    # Exclusion wording creates an upper bound; do not treat every price threshold as a budget.\n    m=re.search(r\'(?:qua|tren|toi da|khong qua)\\s+(\\d+(?:[.,]\\d+)?\\s*(?:nghin|ngan|trieu))\',text)\n    if m and (\'hoa don\' in text or \'cai nao\' in text) and any(x in text for x in (\'de do\',\'bo qua\',\'khong tra\',\'toi da\',\'khong qua\')):\n        max_amount=number(m[1]);out.append(f\'Exclude bills costing more than {max_amount:,.0f} VND.\')\n    if any(x in text for x in (\'a khong\',\'a hong\',\'a thoi\',\'a lon\',\'a mo\',\'khong phai\')):\n        out.append(\'The latest correction overrides the earlier entity, time or action.\')\n    if any(x in text for x in (\'neu\',\'khi\',\'thi thoi\',\'con khong\',\'khong thi\',\'rieng\',\'ngoai tru\',\'qua\',\'duoi\',\'tren\')):\n        out.append(\'Check the requested condition before any dependent write; do not add an unrequested operation.\')\n    return out,days,max_amount\n\ndef bill_evidence(row,trajectory,days,limit):\n    bills=[]\n    for p in trajectory:\n        if p[\'action\'][\'name\']==\'bill_payment.list_bills\':bills=p[\'observation\'].get(\'bills\',[])\n    paid={p[\'action\'][\'arguments\'].get(\'bill_id\') for p in trajectory if p[\'action\'][\'name\']==\'bill_payment.pay\' and p[\'observation\'].get(\'ok\') is True and not p[\'observation\'].get(\'error\')}\n    now=datetime.fromisoformat(row[\'context\'][\'current_time\']).date()\n    eligible=[];excluded=[]\n    for b in bills:\n        if b.get(\'paid\') or b[\'id\'] in paid:continue\n        due=datetime.fromisoformat(b[\'due_date\']).date()\n        reason=\'over the spending limit\' if limit is not None and b[\'amount\']>limit else \'outside the due-date window\' if days is not None and not now<=due<=now+timedelta(days=days) else None\n        if reason:excluded.append((b,reason))\n        else:eligible.append(b)\n    return eligible,excluded\n\nGOALS={\n \'calendar.update_event\':\'Reschedule the existing event rather than creating a duplicate.\',\n \'calendar.cancel_event\':\'Cancel the identified event only if the current request authorizes cancellation.\',\n \'calendar.create_event\':\'Create the requested new event; do not use creation to modify an existing event.\',\n \'reminders.create\':\'Create the requested reminder with the intended content and local time.\',\n \'bill_payment.pay\':\'Pay one eligible unpaid bill through the requested payment source.\',\n \'bill_payment.schedule_autopay\':\'Change recurring payment settings only when explicitly requested.\',\n \'banking.transfer\':\'Transfer the requested amount to the intended beneficiary using the permitted funding account.\',\n \'ewallet.transfer\':\'Transfer the requested amount to the intended recipient, preserving any balance condition.\',\n \'smart_home.light_set\':\'Apply the requested light change to the latest intended room or device.\',\n \'smart_home.fan_set\':\'Apply the requested fan state; switching off is not a request to increase speed.\',\n \'ride_hailing.book\':\'Book the selected ride with the requested pickup, destination and scheduled time.\',\n \'home_automation.create_rule\':\'Create only the requested rule with the correct target, action and local time.\',\n \'sms.send\':\'Send the intended message to the resolved recipient; preserve the requested meaning and facts.\',\n \'email.reply\':\'Reply to the referenced email with the latest corrected message, without inventing details.\',\n \'tasks.complete\':\'Complete the matching pending task; do not substitute another task identifier.\'}\n\nGOALS.update({\n \'weather.get_forecast\':\'Resolve the requested future weather and rain condition before deciding any weather-dependent operation.\',\n \'weather.get_current\':\'Retrieve current weather for the requested place; do not substitute it for a future forecast.\',\n \'bill_payment.list_bills\':\'Identify unpaid bills and inspect their amounts and due dates before selecting eligible payments.\',\n \'banking.get_balance\':\'Inspect available accounts and balances before choosing a funding source or evaluating the requested balance condition.\',\n \'ewallet.get_balance\':\'Inspect the wallet balance before evaluating a payment or fallback-source condition.\',\n \'banking.list_beneficiaries\':\'Resolve the intended beneficiary from the available records; distinguish similar names before selecting an identifier.\',\n \'contacts.search\':\'Resolve the person specified by the latest request or correction, rather than selecting the first similar name.\',\n \'calendar.list_events\':\'Identify the referenced event and its existing timing before deciding the requested change or conflict condition.\',\n \'alarm.list\':\'Resolve the referenced existing alarm and its time before changing or deleting it.\',\n \'tv.guide\':\'Resolve the requested channel or program against the available guide instead of guessing an approximate channel name.\',\n \'gold_price.get_price\':\'Retrieve the requested gold type and buying or selling price to answer the query or evaluate its threshold.\',\n \'fuel_price.get_price\':\'Retrieve the specifically requested fuel type rather than silently using another product.\',\n \'home_security.get_camera\':\'Inspect the camera at the requested location before evaluating any dependent security action.\',\n \'appliances.list\':\'Find the requested appliance type and available controls before selecting a compatible operation.\',\n \'learning.list_courses\':\'Resolve the intended course and enrollment state before scheduling or enrolling.\',\n \'notes.list\':\'Locate the referenced note and inspect its contents before answering or modifying it.\',\n \'tasks.list\':\'Find the referenced tasks and their deadlines or completion state before applying the requested filter.\',\n \'food_delivery.get_menu\':\'Inspect the selected restaurant menu to resolve valid items and user-requested options.\',\n \'loyalty.get_points\':\'Inspect the specified program and available points before evaluating redemption eligibility.\',\n \'loyalty.list_vouchers\':\'Inspect available rewards and their costs before choosing the requested eligible reward.\'})\n\ndef rationale(row,action,trajectory):\n    name=action[\'name\'];args=action[\'arguments\'];rules,days,limit=constraints(row)\n    if name==\'sys.ask_missing_info\':\n        return f"The supported {args[\'tool_name\']} operation still lacks {\', \'.join(args[\'params\'])}. Request that information instead of inventing values."\n    if name==\'bill_payment.pay\' and (days is not None or limit is not None):\n        eligible,excluded=bill_evidence(row,trajectory,days,limit)\n        selected=next((b for b in eligible if b[\'id\']==args.get(\'bill_id\')),None)\n        if selected:\n            other=[b.get(\'type\',b[\'id\']) for b in eligible if b[\'id\']!=selected[\'id\']]\n            parts=[f"Pay the observed {selected.get(\'type\',\'selected\')} bill ({selected[\'id\']}) through {args.get(\'source\',\'the requested source\')}: {selected[\'amount\']:,.0f} VND due {selected[\'due_date\']} satisfies the requested filters."]\n            if other:parts.append(\'Other eligible unpaid bills remain: \'+\', \'.join(other)+\'.\')\n            if excluded:parts.append(\'Leave \'+\', \'.join(b.get(\'type\',b[\'id\'])+\' (\'+reason+\')\' for b,reason in excluded)+\' unchanged.\')\n            return \' \'.join(parts)\n    if name in GOALS:goal=GOALS[name]\n    else:\n        app,verb=name.split(\'.\',1);words=verb.replace(\'_\',\' \')\n        goal=(f\'Look up {verb.partition("_")[2].replace("_"," ") or words} in {app.replace("_"," ")} to resolve the next requested step.\' if verb.startswith((\'get\',\'list\',\'search\',\'find\',\'read\',\'track\')) else\n              f\'Perform the requested {app.replace("_"," ")} operation: {words}. Preserve its user-specified scope.\')\n    facts=[]\n    for field,value in args.items():\n        if field.endswith(\'_id\'):\n            record=next((rec for source in reversed(visible(row,trajectory)) for rec in records(source) if rec.get(\'id\')==value),None)\n            if record:\n                selector=record.get(\'name\',record.get(\'title\',record.get(\'content\',record.get(\'type\',value))))\n                facts.append(f\'Use observed {field}={value} for {json.dumps(str(selector)[:75],ensure_ascii=False)}.\')\n            elif supported(value,row,trajectory):facts.append(f\'Use {field}={value} from the visible request/history.\')\n        elif field in (\'time\',\'start\',\'end\',\'schedule_time\',\'date\'):\n            # Date/time are supervised derived values, not unobserved tool results.\n            facts.append(f\'Resolve {field} as {value} from the requested time and supplied clock.\')\n        elif field in (\'amount\',\'type\',\'source\',\'power\',\'action\',\'brightness\',\'enabled\') and not isinstance(value,(dict,list)):\n            facts.append(f\'Preserve {field}={json.dumps(value,ensure_ascii=False)} for this requested step.\')\n    parts=[goal,*rules[:2],*facts[:3]]\n    if trajectory and trajectory[-1][\'observation\'].get(\'ok\') is True:\n        parts.append(\'Reuse prior results and reassess remaining objectives after this step.\')\n    return \' \'.join(parts)\n\ndef terminal(row,trajectory,specs=None):\n    if not trajectory:return None\n    if trajectory[-1][\'action\'][\'name\']==\'sys.ask_missing_info\':\n        return {\'thought\':\'The missing information was requested. End this turn and wait for the user instead of executing the incomplete operation.\',\'next\':\'summarizer\'}\n    effects=[p for p in trajectory if (specs or {}).get(p[\'action\'][\'name\'],{}).get(\'kind\')==\'write\']\n    if effects:\n        descriptions=[]\n        for p in effects[-4:]:\n            action=p[\'action\'];details=[]\n            for key,value in action[\'arguments\'].items():\n                if key.endswith(\'_id\') or key in (\'time\',\'start\',\'end\',\'amount\',\'power\',\'action\',\'brightness\'):\n                    details.append(f\'{key}={value}\')\n            descriptions.append(action[\'name\']+(\' [\'+\', \'.join(details[:2])+\']\' if details else \'\'))\n        thought=\'The observed results confirm \'+ \'; \'.join(descriptions)+\'. Summarize the completed request without repeating these effects or adding a new operation.\'\n    else:thought=\'The existing lookup results supply the requested information. Answer from those observations instead of repeating an unchanged read or inventing a write.\'\n    return {\'thought\':thought,\'next\':\'summarizer\'}\n\ndef issues(row,trajectory,action,observation):\n    result=[]\n    if observation.get(\'ok\') is False or observation.get(\'error\'):result.append(\'tool_reported_failure\')\n    for key,value in action[\'arguments\'].items():\n        if key.endswith(\'_id\') and isinstance(value,str) and not supported(value,row,trajectory):result.append(\'unobserved_identifier:\'+key)\n    _,days,limit=constraints(row)\n    if action[\'name\']==\'bill_payment.pay\' and (days is not None or limit is not None):\n        eligible,excluded=bill_evidence(row,trajectory,days,limit)\n        if any(b[\'id\']==action[\'arguments\'].get(\'bill_id\') for b,_ in excluded):result.append(\'bill_violates_request_filter\')\n    return result\n', 'umilife/node_eval.py': '"""Post-stage vLLM generation metrics for CE-best and last local checkpoints."""\nimport argparse,json,os,sys,subprocess,time,re,shutil\nfrom pathlib import Path\nfrom collections import Counter\nfrom jsonschema import Draft202012Validator,ValidationError\nfrom umilife.core import read_jsonl,write_json,parse_json,load_catalog,digest as file_digest\nfrom umilife.prompt_policy import request_schema,digest\nfrom umilife.evaluate import matches\n\ndef assess(role,record,raw,catalog):\n    target=parse_json(record[\'target\']);messages=json.loads(record[\'messages\'][1][\'content\'])\n    if role==\'planner\':\n        names=re.findall(r\'^\\s*"([^"\\n]+)"\\(args:\',messages[\'tools_ts\'],re.M)\n        specs={name:catalog[name] for name in names}\n        schema=request_schema(\'planner\',specs)\n    else:schema=request_schema(\'caller\',{target[\'name\']:catalog[target[\'name\']]},{\'tool\':target[\'name\']})\n    value=None;valid=False;error=None\n    try:\n        value=parse_json(raw);Draft202012Validator(schema).validate(value)\n        if role==\'planner\' and value.get(\'next\')==\'caller\' and (not value.get(\'tool\') or value[\'tool\'] not in catalog):raise ValueError(\'Missing/unknown selected tool\')\n        valid=True\n    except (ValueError,ValidationError) as exc:error=str(exc)[:300]\n    result={\'schema_valid\':valid,\'schema_error\':error,\'parsed\':value,\'target\':target}\n    if role==\'planner\':\n        route=bool(valid and value[\'next\']==target[\'next\']);tool=bool(valid and value.get(\'tool\')==target.get(\'tool\')) if target[\'next\']==\'caller\' else None\n        result.update(correct=bool(route and (target[\'next\']!=\'caller\' or tool)),route_match=route,tool_match=tool,terminal=target[\'next\']!=\'caller\')\n    else:\n        correct=bool(valid and matches(value,target,catalog));fields={}\n        pred=value.get(\'arguments\',{}) if valid else {}\n        for k,v in target[\'arguments\'].items():\n            fields[k]=bool(valid and k in pred and matches({\'name\':value[\'name\'],\'arguments\':{k:pred[k]}},{\'name\':target[\'name\'],\'arguments\':{k:v}},catalog))\n        count=sum(fields.values());denom=len(pred)+len(target[\'arguments\'])\n        result.update(correct=correct,field_matches=fields,argument_f1=2*count/denom if denom else float(valid and value[\'name\']==target[\'name\']))\n    return result\n\ndef summarize(rows,role):\n    n=len(rows);assert n\n    report={\'samples\':n,\'match_accuracy\':sum(r[\'correct\'] for r in rows)/n,\'schema_valid_rate\':sum(r[\'schema_valid\'] for r in rows)/n}\n    if role==\'planner\':\n        report[\'route_accuracy\']=sum(r[\'route_match\'] for r in rows)/n\n        for kind in (\'initial\',\'intermediate\',\'terminal\'):\n            sub=[r for r in rows if (\'terminal\' if r[\'terminal\'] else \'initial\' if r[\'step\']==0 else \'intermediate\')==kind]\n            report[kind]={\'n\':len(sub),\'accuracy\':sum(r[\'correct\'] for r in sub)/len(sub) if sub else None}\n        report[\'continued_after_reference_end\']=sum(r[\'terminal\'] and isinstance(r[\'parsed\'],dict) and r[\'parsed\'].get(\'next\')==\'caller\' for r in rows)\n    else:\n        fields={}\n        for row in rows:\n            for k,v in row[\'field_matches\'].items():fields.setdefault(k,{\'n\':0,\'correct\':0});fields[k][\'n\']+=1;fields[k][\'correct\']+=v\n        report.update(argument_f1=sum(r[\'argument_f1\'] for r in rows)/n,fields=fields)\n    return report\n\ndef worker(a):\n    from umilife.t4_compat import prepare_t4_lora_kernel\n    print(json.dumps(prepare_t4_lora_kernel()),flush=True)\n    import torch\n    from vllm import LLM,SamplingParams\n    from vllm.lora.request import LoRARequest\n    from umilife.runtime_options import engine_options\n    work=Path(a.output);config=json.loads((work/\'config.json\').read_text());options=engine_options(torch.cuda.get_device_capability(0))\n    engine=LLM(model=a.model,revision=a.revision,tokenizer=config[\'tokenizer\'],dtype=\'float16\',tensor_parallel_size=1,\n        enable_lora=True,max_loras=2,max_cpu_loras=2,max_lora_rank=16,max_model_len=8192,max_num_seqs=4,max_num_batched_tokens=8192,\n        gpu_memory_utilization=.85,enforce_eager=not options[\'cuda_graphs\'],enable_prefix_caching=options[\'prefix_caching\'],\n        enable_chunked_prefill=False,seed=42,generation_config=\'vllm\',disable_log_stats=True)\n    tokenizer=engine.get_tokenizer();records=read_jsonl(Path(a.prepared)/f\'{a.stage}.validation.jsonl\')[a.rank::2]\n    catalog=load_catalog(a.data_root,\'train\');sampling=SamplingParams(temperature=0,seed=42,max_tokens=512,guided_decoding=None)\n    candidates={name:LoRARequest(name,i+1,path) for i,(name,path) in enumerate(config[\'candidates\'].items())}\n    target=work/f\'gpu{a.rank}.jsonl\'\n    with target.open(\'w\') as out:\n        for start in range(0,len(records),4):\n            group=records[start:start+4]\n            prompts=[tokenizer.apply_chat_template(r[\'messages\'],tokenize=False,add_generation_prompt=True) for r in group]\n            token_ids=[tokenizer.encode(p,add_special_tokens=False) for p in prompts]\n            assert all(len(ids)+512<=8192 for ids in token_ids),\'Context exceeded; no evidence truncation\'\n            order=list(candidates)\n            if start//4%2:order.reverse()\n            for name in order:\n                tick=time.monotonic();results=engine.generate(prompts,sampling,lora_request=candidates[name],use_tqdm=False)\n                for record,ids,result in zip(group,token_ids,results,strict=True):\n                    row={\'id\':record[\'id\'],\'step\':record[\'step\'],\'candidate\':name,\'prompt_sha256\':digest(record[\'messages\']),\n                        \'input_ids_sha256\':digest(ids),\'raw\':result.outputs[0].text,\'finish_reason\':result.outputs[0].finish_reason,\n                        **assess(a.stage,record,result.outputs[0].text,catalog)}\n                    out.write(json.dumps(row,ensure_ascii=False)+\'\\n\');out.flush()\n                print(json.dumps({\'stage\':a.stage,\'rank\':a.rank,\'candidate\':name,\'completed\':min(start+4,len(records)),\n                                  \'total\':len(records),\'batch_seconds\':time.monotonic()-tick}),flush=True)\n\ndef export_adapter(adapter,weights,export,lineage,report):\n    # Export a self-contained inference adapter without overwriting the CE-best training artifact.\n    temp=export.with_name(export.name+\'.tmp\')\n    if temp.exists():shutil.rmtree(temp)\n    temp.mkdir(parents=True)\n    for path in adapter.iterdir():\n        if path.is_file() and path.name not in (\'adapter_model.safetensors\',\'lineage.json\'):shutil.copy2(path,temp/path.name)\n    shutil.copy2(weights,temp/weights.name)\n    import torch\n    from safetensors import safe_open\n    parameters=0\n    with safe_open(weights,framework=\'pt\',device=\'cpu\') as handle:\n        for key in handle.keys():\n            value=handle.get_tensor(key);assert torch.isfinite(value).all(),key;parameters+=value.numel()\n    selected_lineage={**lineage,\'selection\':report,\n        \'ce_best_final_adapter\':lineage[\'final_adapter\'],\n        \'final_adapter\':{\'sha256\':file_digest(weights),\'hash_kind\':\'safetensors_file\',\'finite\':True,\'parameters\':parameters}}\n    write_json(temp/\'lineage.json\',selected_lineage)\n    if export.exists():shutil.rmtree(export)\n    temp.rename(export)\n    return export\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'stage\',\'prepared\',\'adapter\',\'checkpoints\',\'data-root\',\'model\',\'revision\',\'output\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--rank\',type=int)\n    a=p.parse_args()\n    if a.rank is not None:worker(a);return\n    work=Path(a.output);work.mkdir(parents=True,exist_ok=True);adapter=Path(a.adapter);cp=Path(a.checkpoints)\n    latest=cp/json.loads((cp/\'latest.json\').read_text())[\'folder\']\n    lineage=json.loads((adapter/\'lineage.json\').read_text())\n    assert lineage[\'stage\']==a.stage and lineage[\'model\']==a.model and lineage[\'revision\']==a.revision\n    assert lineage[\'dataset_manifest_sha256\']==file_digest(Path(a.prepared)/\'manifest.json\')\n    candidates={\'ce_best\':str(adapter),\'last\':str(latest)}\n    if file_digest(adapter/\'adapter_model.safetensors\')==file_digest(latest/\'adapter_model.safetensors\'):candidates.pop(\'last\')\n    config={\'stage\':a.stage,\'model\':a.model,\'revision\':a.revision,\'tokenizer\':str(adapter),\'candidates\':candidates,\n        \'weights_sha256\':{k:file_digest(Path(v)/\'adapter_model.safetensors\') for k,v in candidates.items()},\n        \'manifest_sha256\':file_digest(Path(a.prepared)/\'manifest.json\'),\'prompt_sha256\':json.loads((Path(a.prepared)/\'manifest.json\').read_text())[\'prompts_sha256\']}\n    if (work/\'report.json\').exists():\n        assert json.loads((work/\'config.json\').read_text())==config,\'Stale node eval output\'\n        print((work/\'report.json\').read_text());return\n    write_json(work/\'config.json\',config)\n    processes=[];logs=[];last_seen=[None,None]\n    try:\n        for rank in range(2):\n            env=os.environ.copy();env.update(CUDA_VISIBLE_DEVICES=str(rank),TRITON_CACHE_DIR=f\'/tmp/node_eval_triton_{rank}\')\n            log=(work/f\'gpu{rank}.log\').open(\'w\');logs.append(log)\n            command=[sys.executable,\'-u\',\'-m\',\'umilife.node_eval\',*sys.argv[1:],\'--rank\',str(rank)]\n            processes.append(subprocess.Popen(command,env=env,stdout=log,stderr=subprocess.STDOUT))\n        while any(p.poll() is None for p in processes):\n            if any(p.poll() not in (None,0) for p in processes):raise RuntimeError(\'Node generation worker failed\')\n            for rank,log in enumerate(logs):\n                log.flush()\n                for line in reversed((work/f\'gpu{rank}.log\').read_text().splitlines()[-12:]):\n                    try:event=json.loads(line)\n                    except ValueError:continue\n                    if isinstance(event,dict) and \'completed\' in event:\n                        if line!=last_seen[rank]:print(json.dumps({\'node_eval_progress\':event}),flush=True);last_seen[rank]=line\n                        break\n            time.sleep(5)\n        assert all(p.returncode==0 for p in processes),\'Node generation worker failed\'\n    except BaseException:\n        for process in processes:\n            if process.poll() is None:process.terminate()\n        for rank in range(len(logs)):\n            logs[rank].flush();print((work/f\'gpu{rank}.log\').read_text()[-6000:],flush=True)\n        raise\n    finally:\n        for process in processes:\n            if process.poll() is None:\n                try:process.wait(timeout=20)\n                except subprocess.TimeoutExpired:process.kill();process.wait()\n        for log in logs:log.close()\n    rows=[r for rank in range(2) for r in read_jsonl(work/f\'gpu{rank}.jsonl\')]\n    expected={(r[\'id\'],r[\'step\']) for r in read_jsonl(Path(a.prepared)/f\'{a.stage}.validation.jsonl\')}\n    for name in candidates:\n        selected=[r for r in rows if r[\'candidate\']==name]\n        assert len(selected)==len(expected) and {(r[\'id\'],r[\'step\']) for r in selected}==expected\n    for key in expected:\n        pair=[r for r in rows if (r[\'id\'],r[\'step\'])==key]\n        assert len({(r[\'prompt_sha256\'],r[\'input_ids_sha256\']) for r in pair})==1\n    metrics={name:summarize([r for r in rows if r[\'candidate\']==name],a.stage) for name in candidates}\n    winner=max(metrics,key=lambda k:(metrics[k][\'match_accuracy\'],metrics[k][\'schema_valid_rate\'],k==\'ce_best\'))\n    report={\'stage\':a.stage,\'metrics\':metrics,\'recommended_checkpoint\':winner,\'recommended_path\':candidates[winner],\n            \'guided_decoding\':False,\'batch_per_gpu\':4,\'gpu_workers\':2,\n            \'note\':\'Reference-prefix node agreement, not full workflow/contest score. CE-best and last only; no claim of globally best checkpoint.\'}\n    export=work.parent.parent/\'selected_adapters\'/a.stage\n    export_adapter(adapter,Path(candidates[winner])/\'adapter_model.safetensors\',export,lineage,report)\n    report[\'exported_adapter\']=str(export)\n    write_json(work/\'report.json\',report);print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'umilife/prepare_2gpu.py': '"""Fixed split; train-only evidence/repair supervision and explicit label quarantine."""\nimport argparse,json,hashlib,copy\nfrom pathlib import Path\nfrom collections import Counter\nfrom jsonschema import Draft202012Validator\nfrom umilife.core import read_jsonl,write_json,dumps,digest,allowed_specs,load_catalog,import_toolenv\nfrom umilife.data import normalized_row\nfrom umilife.train import encode\nfrom umilife.prompt_policy import build_messages,request_schema,POLICY_AUDIT\nfrom umilife.evidence_labels import rationale,terminal,issues,VERSION\nROLES=(\'planner\',\'caller\')\nLEAN=\'Carry out the selected requested operation using the original request, history and observed results. Preserve all conditions, corrections and intent-required fields.\'\n\ndef prepare(root,out,split_file,tokenizer):\n    root=Path(root);out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    assert digest(root/\'train/train.jsonl\')==\'24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8\'\n    valid=set(Path(split_file).read_text().splitlines());assert len(valid)==245\n    rows=read_jsonl(root/\'train/train.jsonl\');assert len(rows)==1545 and valid<={r[\'id\'] for r in rows}\n    zero=json.loads((Path(split_file).parent/\'zero_action_train_labels.json\').read_text())\n    zero=zero[\'labels\'];assert not valid&zero.keys(),\'Training annotations must not include validation IDs\'\n    catalog=load_catalog(root,\'train\');Env=import_toolenv(root,\'train\')\n    counts=Counter();audit=[];caches={(role,part):[] for role in ROLES for part in (\'train\',\'validation\')};stats={};references=[]\n    handles={key:(out/f\'{key[0]}.{key[1]}.jsonl\').open(\'w\') for key in caches}\n    def add(role,raw,specs,trajectory,target,plan=None,feedback=None,kind=\'reference\'):\n        part=\'validation\' if raw[\'id\'] in valid else \'train\';key=(role,part)\n        schema=request_schema(role,specs,plan);Draft202012Validator(schema).validate(target)\n        rec={\'id\':raw[\'id\'],\'role\':role,\'step\':len(trajectory),\'kind\':kind,\n             \'messages\':build_messages(role,normalized_row(raw),raw[\'audio_text\'],specs,trajectory,plan,feedback),\'target\':dumps(target)}\n        encoded=encode(tokenizer,rec);assert len(encoded[\'input_ids\'])<=8192,(\'Overlength\',raw[\'id\'],role)\n        # Targets and diagnostic metadata are separate from message construction.\n        handles[key].write(dumps(rec)+\'\\n\');caches[key].append(encoded);counts[f\'{role}.{part}\']+=1;counts[\'kind.\'+kind+\'.\'+part]+=1\n    try:\n        for raw in rows:\n            part=\'validation\' if raw[\'id\'] in valid else \'train\';specs=allowed_specs(raw,catalog)\n            env=Env(\'train\',raw[\'id\'],root=root);trajectory=[];steps=[];problems=[]\n            actions=raw[\'label\'][\'tool_calls\'];observations=raw[\'label\'][\'tool_results\'];assert len(actions)==len(observations)\n            for index,(action,expected) in enumerate(zip(actions,observations,strict=True)):\n                assert env.validate(action[\'name\'],action[\'arguments\']).get(\'ok\')\n                observed=env.call(action[\'name\'],action[\'arguments\']);assert observed==expected,(raw[\'id\'],index)\n                found=issues(raw,trajectory,action,observed)\n                if found:problems.append({\'step\':index,\'issues\':found})\n                plan={\'thought\':rationale(raw,action,trajectory),\'next\':\'caller\',\'tool\':action[\'name\']}\n                steps.append((copy.deepcopy(trajectory),action,plan))\n                trajectory.append({\'step\':index,\'thought\':plan[\'thought\'],\'action\':copy.deepcopy(action),\'observation\':copy.deepcopy(observed)})\n            references.append({\'id\':raw[\'id\'],\'tool_calls\':actions,\'state_hash\':env.get_state_hash()})\n            if problems:\n                audit.append({\'id\':raw[\'id\'],\'split\':part,\'excluded_from_train\':part==\'train\',\'problems\':problems})\n                if part==\'train\':continue\n            if not actions:\n                if part==\'train\':add(\'planner\',raw,specs,[],zero[raw[\'id\']],kind=\'reviewed_no_action\')\n                else:audit.append({\'id\':raw[\'id\'],\'split\':part,\'excluded_from_train\':False,\'unscored\':\'No explicit terminal route in contest annotation\'})\n                continue\n            used_repair=False;used_read=False\n            for prior,action,plan in steps:\n                add(\'planner\',raw,specs,prior,plan)\n                caller_plan=plan\n                # Half of train Caller records and ALL valid records get a non-oracle generic plan.\n                if part==\'validation\' or int(hashlib.sha256(f"{raw[\'id\']}:{len(prior)}".encode()).hexdigest()[:8],16)%2==0:\n                    caller_plan={**plan,\'thought\':LEAN}\n                add(\'caller\',raw,specs,prior,action,caller_plan)\n                if part!=\'train\':continue\n                if prior and not used_read:\n                    last=prior[-1][\'action\']\n                    if specs[last[\'name\']].get(\'kind\')==\'read\' and last[\'name\']!=action[\'name\']:\n                        feedback={\'kind\':\'repeated_read\',\'action\':last,\'instruction\':\'Reuse the existing unchanged result and advance the pending objective.\'}\n                        if int(hashlib.sha256(raw[\'id\'].encode()).hexdigest()[:8],16)%2:\n                            feedback={\'kind\':\'completion_review\',\'proposal\':{\'thought\':\'The request may be complete.\',\'next\':\'summarizer\'},\'instruction\':\'Check every requested objective against current observations. Continue if an authorized objective remains.\'}\n                            corrected={**plan,\'thought\':\'An authorized objective remains; do not end after lookup alone. \'+plan[\'thought\']};kind=\'premature_end_recovery\'\n                        else:corrected={**plan,\'thought\':\'Reuse the previous lookup; it already supplies evidence. \'+plan[\'thought\']};kind=\'read_recovery\'\n                        add(\'planner\',raw,specs,prior,corrected,feedback=feedback,kind=kind);used_read=True\n                if not used_repair:\n                    optional=[k for k in action[\'arguments\'] if k not in specs[action[\'name\']][\'parameters\'].get(\'required\',[])]\n                    if optional:\n                        proposal=copy.deepcopy(action);proposal[\'arguments\'].pop(optional[0])\n                        feedback={\'kind\':\'intent_review\',\'proposal\':proposal,\'instruction\':\'Check whether the proposed call omits a field needed by the original request. Restore intent-required information from available evidence, even if the API marks the field optional.\'}\n                        add(\'caller\',raw,specs,prior,action,{**plan,\'thought\':LEAN},feedback=feedback,kind=\'argument_repair\');used_repair=True\n            add(\'planner\',raw,specs,trajectory,terminal(raw,trajectory,specs),kind=\'terminal\')\n    finally:\n        for f in handles.values():f.close()\n    import torch\n    files={p.name:digest(p) for p in out.glob(\'*.jsonl\')}\n    for (role,part),items in caches.items():\n        stat={\'loaded\':len(items),\'used\':len(items),\'overlength\':0,\'max_tokens\':max((len(i[\'input_ids\']) for i in items),default=0),\n              \'max_target_tokens\':max((sum(v!=-100 for v in i[\'labels\']) for i in items),default=0)}\n        path=out/f\'{role}.{part}.jsonl\';torch.save({\'items\':items,\'stats\':stat},out/f\'{role}.{part}.{files[path.name][:16]}.tokens.pt\');stats[f\'{role}.{part}\']=stat\n    manifest={\'train_episodes\':1300,\'validation_episodes\':245,\'split\':{r[\'id\']:\'validation\' if r[\'id\'] in valid else \'train\' for r in rows},\n        \'counts\':dict(counts),\'prompts_sha256\':POLICY_AUDIT[\'static_prompts_sha256\'],\'rationale_authors\':[\'Codex\'],\n        \'rationale_provenance\':\'Codex-authored evidence compiler \'+VERSION+\' plus100 manually reviewed training-only no-action targets; not independent per-example LLM reasoning.\',\n        \'rationale_sha256\':digest(Path(__file__).with_name(\'evidence_labels.py\')),\'zero_labels_sha256\':digest(Path(split_file).parent/\'zero_action_train_labels.json\'),\n        \'train_quarantined_episodes\':sum(x.get(\'excluded_from_train\',False) for x in audit),\n        \'validation_unscored_no_action\':sum(\'unscored\' in x for x in audit),\'tokenization\':stats,\'files_sha256\':files,\n        \'split_sha256\':digest(split_file),\'model_input\':\'Clean provided train transcripts; no ASR augmentation. Reference prefix only; repair augmentation train-only.\',\n        \'validation_usage\':\'Reference-only CE and generation metrics. No optimizer update on valid; no validation-derived prompt exceptions.\'}\n    write_json(out/\'manifest.json\',manifest);write_json(out/\'data_audit.json\',audit);write_json(out/\'references.json\',references)\n    print(dumps(manifest),flush=True);return manifest\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'output\',\'split-file\',\'model\',\'revision\'):p.add_argument(\'--\'+name,required=True)\n    a=p.parse_args();from transformers import AutoTokenizer\n    prepare(a.data_root,a.output,a.split_file,AutoTokenizer.from_pretrained(a.model,revision=a.revision))\n', 'umilife/prompt_policy.py': '"""Generic tool policy. No benchmark examples, held-out labels or per-sample rules."""\nfrom .typescript_schema import typescript_catalog\nfrom copy import deepcopy\nfrom datetime import datetime,timedelta\nimport hashlib,json\n\nPOLICY_VERSION=\'planner-caller-evidence-v3\'\nCOMMON="""You are LifeMate, a personal assistant using installed tools.\nAct only on the current request. History resolves references and may satisfy explicit already-completed or recall conditions, but is not a newly executed operation in this turn. Preserve quantities, negation, conditions and exact text to send or save. Use the supplied clock and timezone. Never invent identifiers or tool results.\nOnly current observations confirm execution. Do not repeat successful effects, even with reworded arguments. Retry only with evidence of failure and a justified correction. Reuse unchanged read results; change strategy after an unhelpful lookup.\nTool contracts are TypeScript declarations: ? marks optional fields, literal unions specify enums, and comments describe constraints and identifier sources. These are interface documents, not executable code. Return one JSON object without markdown.\n"""\nPROMPTS={\n \'planner\':COMMON+"""You are Planner. Decide the next single step toward pending user objectives.\nChoose the installed tool, order dependent operations, evaluate conditions from available evidence, and stop when no further authorized operation is needed. If a condition or required identifier needs discovery, select an appropriate read tool first. Choose sys.ask_missing_info when required information cannot be resolved from context or supported lookups. An unresolved ID alone is not an unsupported request.\nWrite thought in English, normally one to three sentences. Identify the pending objective, decisive user constraints and the visible evidence supporting the next step. Explain why the selected operation is appropriate, not just its name or parameter sources. Keep selection predicates, negation and the latest correction. You may state observed identifiers or derived quantities/times when justified; do not invent evidence or claim an unexecuted result. Resolve business conditions and choose the intended entity; Caller binds that decision to the selected contract and verifies values against the original request. Distinguish modifying an existing object from creating another one. Track which objectives were completed and what remains. End only with supporting observations or an explicit no-op/unsupported reason. Reuse a successful unchanged lookup after repeated-read feedback; repeating its name is not progress.\nFor an operation return {thought,next:"caller",tool}. For a completed/no-op turn or after requesting clarification return {thought,next:"summarizer"}; this terminal route does not require a trained Summarizer. For an unsupported request return {thought,next:"give_up"}. Never stop while a supported, authorized objective remains, repeat a confirmed write for a reply, or claim unconfirmed success.\n""",\n \'caller\':COMMON+"""You are Caller. Convert Planner\'s selected step into one valid API call.\nReturn exactly {name,arguments}; name must equal plan.tool. Planner owns routing and tool selection. Resolve values using the current request, history, context and previous tool observations. Treat plan.thought as guidance, not proof. Match the intended entity before taking its ID; use identifiers actually observed or explicitly supplied. Respect user qualifiers rather than choosing the first result. Normalize dates with the supplied clock/timezone and preserve exact text to send or save.\nRead the selected TypeScript contract, comments, required/optional fields, enums and alternative argument groups. Supply only supported values. Optional in the API does not mean optional for the user intent: include every field needed to express the requested schedule, end time, filter, type or state. Omit optional fields only when they have no supporting requirement. A latest correction overrides the earlier value. Distinguish off from changing speed/brightness. Convert relative dates and clock expressions using the supplied clock/timezone; verify arithmetic rather than copying an unrelated time. Preserve the requested message facts and do not invent recipients or numbers. For sys.ask_missing_info, use the intended tool_name and exact missing parameter names from its contract. Never invent a required identifier or change the selected tool. Correct validation feedback without repeating confirmed effects. Output no thought, next or answer. The runtime executes your JSON.\n""",\n \'summarizer\':COMMON+"Return only {answer}, briefly grounded in actual observations. Do not claim unconfirmed success.",\n \'global\':COMMON+"Handle one ReAct step: return {thought,next,tool,action} for caller, {thought,next,answer} for summarizer, or {thought,next} for give_up. Keep the English thought concise; action contains {name,arguments}."\n}\nCOMPLETION_REVIEW="""Review this proposed ending against every objective in the current request. Has something only been looked up but not executed? Are you confusing history with current execution? Are there unfinished objectives, actual operation failures, or unresolved empty searches? Do not mistake business-state fields for execution failures. Do not propose repeating any confirmed effect just to verify it or generate a reply; summarize when all objectives are satisfied. A different argument spelling does not justify repeating a completed objective. Continue with caller when work remains. Prefer lookup before clarification. Summarize only with sufficient evidence or while waiting for clarification; give_up only beyond installed capabilities."""\n\ndef canonical(value):return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))\ndef digest(value):return hashlib.sha256(canonical(value).encode()).hexdigest()\ndef obj(properties,required):return {\'type\':\'object\',\'properties\':properties,\'required\':required,\'additionalProperties\':False}\n\ndef parameter_schema(schema):\n    """Convert documented x-one-of to ordinary JSON Schema; preserve typed nesting."""\n    allowed={\'type\',\'properties\',\'required\',\'items\',\'enum\',\'format\',\'minimum\',\'maximum\',\n             \'exclusiveMinimum\',\'exclusiveMaximum\',\'minLength\',\'maxLength\',\'minItems\',\'maxItems\',\n             \'anyOf\',\'oneOf\',\'allOf\',\'additionalProperties\',\'description\'}\n    out={}\n    for key,value in schema.items():\n        if key not in allowed:continue\n        if key==\'properties\':out[key]={name:parameter_schema(prop) for name,prop in value.items()}\n        elif key==\'items\' and isinstance(value,dict):out[key]=parameter_schema(value)\n        elif key in (\'anyOf\',\'oneOf\',\'allOf\'):out[key]=[parameter_schema(x) for x in value]\n        else:out[key]=deepcopy(value)\n    if out.get(\'type\')==\'object\' and \'properties\' in out:out[\'additionalProperties\']=False\n    if schema.get(\'x-one-of\'):\n        # Full object alternatives are supported by both JSON Schema and LMFE.\n        base=out\n        out={\'anyOf\':[{**deepcopy(base),\'required\':list(dict.fromkeys(base.get(\'required\',[])+list(group)))}\n                      for group in schema[\'x-one-of\']]}\n    return out\n\ndef action_schema(specs,name):\n    arguments=parameter_schema(specs[name][\'parameters\'])\n    if name==\'sys.ask_missing_info\':\n        # Only public installed schemas provide these choices, never an answer label.\n        choices=[n for n,s in specs.items() if n!=name and s[\'parameters\'].get(\'properties\')]\n        if choices:arguments[\'properties\'][\'tool_name\'][\'enum\']=sorted(choices)\n        arguments[\'properties\'][\'params\'][\'minItems\']=1\n        fields=sorted({key for target in choices for key in specs[target][\'parameters\'][\'properties\']})\n        if fields:arguments[\'properties\'][\'params\'][\'items\'][\'enum\']=fields\n    return obj({\'name\':{\'type\':\'string\',\'enum\':[name]},\'arguments\':arguments},[\'name\',\'arguments\'])\n\ndef route_schema(specs,global_mode=False):\n    thought={\'type\':\'string\',\'maxLength\':1024};branches=[]\n    if specs:\n        if global_mode:\n            for name in sorted(specs):\n                branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                    \'tool\':{\'type\':\'string\',\'enum\':[name]},\'action\':action_schema(specs,name)},[\'thought\',\'next\',\'tool\',\'action\']))\n        else:\n            branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                \'tool\':{\'type\':\'string\',\'enum\':sorted(specs)}},[\'thought\',\'next\',\'tool\']))\n    done={\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'summarizer\']}}\n    if global_mode:done[\'answer\']={\'type\':\'string\',\'maxLength\':1024}\n    branches.append(obj(done,list(done)))\n    branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'give_up\']}},[\'thought\',\'next\']))\n    return {\'anyOf\':branches}\n\ndef compact_catalog(specs):\n    fields=(\'type\',\'enum\',\'x-ref\',\'format\',\'items\',\'minimum\',\'maximum\')\n    return [{\'name\':name,\'kind\':spec.get(\'kind\'),\'description\':spec[\'description\'],\n             \'parameters\':{key:{k:deepcopy(v) for k,v in prop.items() if k in fields}\n                           for key,prop in spec[\'parameters\'].get(\'properties\',{}).items()},\n             \'required\':spec[\'parameters\'].get(\'required\',[]),\n             **({\'one_of\':spec[\'parameters\'][\'x-one-of\']} if spec[\'parameters\'].get(\'x-one-of\') else {})}\n            for name,spec in sorted(specs.items())]\n\ndef reference_clock(context):\n    result={}\n    try:\n        now=datetime.fromisoformat(context[\'current_time\'])\n        result={\'today\':now.date().isoformat(),\'tomorrow\':(now+timedelta(days=1)).date().isoformat(),\n                \'day_after_tomorrow\':(now+timedelta(days=2)).date().isoformat(),\n                \'weekday_iso\':now.isoweekday(),\'timezone\':context.get(\'timezone\')}\n    except (KeyError,TypeError,ValueError):pass\n    return result\n\ndef build_messages(role,row,text,specs,trajectory,plan=None,feedback=None):\n    # Explicit provenance allowlist. Current labels/final state/reference transcript never enter here.\n    context={k:deepcopy(row[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in row[\'context\']}\n    history=[{k:deepcopy(turn[k]) for k in (\'role\',\'text\',\'tool_calls\',\'tool_results\') if k in turn}\n             for turn in row.get(\'history\',[])]\n    content={\'context\':context,\'reference_clock\':reference_clock(context),\'history\':history,\n             \'request\':text,\'trajectory\':deepcopy(trajectory)}\n    if role in (\'planner\',\'global\'):content[\'tools_ts\']=typescript_catalog(specs)\n    elif role==\'caller\':\n        selected=(plan or {}).get(\'tool\');assert selected in specs,\'Planner selected an unavailable tool\'\n        content.update(plan=deepcopy(plan),selected_tool_ts=typescript_catalog({selected:specs[selected]},detailed=True))\n        if selected==\'sys.ask_missing_info\':\n            content[\'missing_information_tools_ts\']=typescript_catalog({name:spec for name,spec in specs.items()\n                if name!=selected and spec[\'parameters\'].get(\'properties\')})\n    else:content[\'plan\']=deepcopy(plan)\n    if feedback is not None:content[\'validation_feedback\']=deepcopy(feedback)\n    # Keep the original request at the end, after schemas/observations/feedback.\n    content.pop(\'request\');content[\'request\']=text\n    return [{\'role\':\'system\',\'content\':PROMPTS[role]},\n            {\'role\':\'user\',\'content\':json.dumps(content,ensure_ascii=False,separators=(\',\',\':\'))}]\n\ndef request_schema(role,specs,plan=None):\n    if role==\'caller\':return action_schema(specs,plan[\'tool\'])\n    if role in (\'planner\',\'global\'):return route_schema(specs,role==\'global\')\n    return obj({\'answer\':{\'type\':\'string\',\'maxLength\':1024}},[\'answer\'])\n\ndef validate_meta_action(action,specs):\n    if action[\'name\']!=\'sys.ask_missing_info\':return None\n    args=action[\'arguments\'];target=args.get(\'tool_name\');params=args.get(\'params\',[])\n    if target not in specs or target==\'sys.ask_missing_info\':return \'Clarification target must be another installed tool\'\n    # An optional API parameter can still be necessary to express the user\'s intent.\n    fields=set(specs[target][\'parameters\'].get(\'properties\',{}))\n    if not params or len(params)!=len(set(params)) or not set(params)<=fields:\n        return \'params must contain unique parameter names from the target tool contract\'\n    return None\n\ndef repeated_read(action,specs,trajectory):\n    if specs[action[\'name\']].get(\'kind\')!=\'read\':return False\n    for old in reversed(trajectory):\n        if specs.get(old[\'action\'][\'name\'],{}).get(\'kind\')==\'write\':return False\n        if canonical(old[\'action\'])==canonical(action):return True\n    return False\n\nPOLICY_AUDIT={\'policy_version\':POLICY_VERSION,\'static_prompts_sha256\':digest(PROMPTS),\n    \'completion_review_sha256\':digest(COMPLETION_REVIEW),\'few_shot_examples\':0,\n    \'sample_specific_rules\':False,\'gold_inputs_to_model\':False,\n    \'allowed_input_sources\':[\'current ASR request\',\'ASR user history\',\'provided context\',\n        \'installed public tool schemas\',\'observations from executed tools\',\'model-generated plan/feedback\']}\n', 'umilife/runtime_options.py': '"""Pinned vLLM 0.9.2 / Triton3.3 compatibility policy, independent of Torch."""\ndef engine_options(capability):\n    # SM75 prefix-prefill fails compilation on the observed FP16 path. Keep graphs.\n    prefix=tuple(capability)>=(8,0)\n    return {\'cuda_graphs\':True,\'prefix_caching\':prefix,\n            \'prefix_caching_policy\':\'sm80_plus\',\n            \'prefix_caching_reason\':(\'enabled_on_sm80_plus\' if prefix else\n                \'disabled_on_pre_sm80_due_to_prefix_prefill_triton_conversion_failure\'),\n            \'compute_capability\':list(capability)}\n', 'umilife/t4_compat.py': '"""Pinned vLLM0.9.2 T4 LoRA kernel compatibility, shared with validated inference."""\ndef prepare_t4_lora_kernel():\n    """Work around Triton 3.3 fp16 conversion failure on SM75; preserve FP32 accumulation."""\n    import torch,importlib.util,importlib.metadata,hashlib\n    path=Path(importlib.util.find_spec("vllm").origin).parent/"lora/ops/triton_ops/kernel_utils.py"\n    original="        accumulator += tl.dot(\\n            tiled_a,\\n            tiled_b,\\n        )"\n    patched="        # LifeMate SM75 compatibility: explicit exact fp16->fp32 before dot.\\n        accumulator += tl.dot(tiled_a.to(tl.float32), tiled_b.to(tl.float32), input_precision=\\"ieee\\")"\n    source=path.read_text()\n    capability=torch.cuda.get_device_capability(0)\n    if capability[0]>=8:\n        if patched in source:path.write_text(source.replace(patched,original))\n        return {"applied":False,"capability":list(capability)}\n    assert importlib.metadata.version("vllm")=="0.9.2"\n    assert importlib.metadata.version("triton")=="3.3.0"\n    assert original in source or patched in source,"Unexpected LoRA kernel source"\n    if original in source:\n        import tempfile\n        with tempfile.NamedTemporaryFile(mode=\'w\',dir=path.parent,delete=False,suffix=\'.py\') as tmp:\n            tmp.write(source.replace(original,patched));temporary=tmp.name\n        os.replace(temporary,path)\n    # Exercise the exact patched helper on the GPU against an FP32 PyTorch reference.\n    global triton,tl,mm_k\n    import triton\n    import triton.language as tl\n    from vllm.lora.ops.triton_ops.kernel_utils import mm_k\n    @triton.jit\n    def check_kernel(A,B,C,K:tl.constexpr):\n        rows=tl.arange(0,32)\n        cols=tl.arange(0,16)\n        kk=tl.arange(0,32)\n        a=A+rows[:,None]*K+kk[None,:]\n        b=B+kk[:,None]*16+cols[None,:]\n        acc=mm_k(a,b,1,16,kk,K,32,16,32,True,1,False,tl.float16)\n        tl.store(C+rows[:,None]*16+cols[None,:],acc)\n    errors=[]\n    torch.manual_seed(42)\n    for k in (128,2560):\n        a=torch.randn(32,k,device="cuda",dtype=torch.float16)\n        b=torch.randn(k,16,device="cuda",dtype=torch.float16)\n        c=torch.empty(32,16,device="cuda",dtype=torch.float32)\n        check_kernel[(1,)](a,b,c,k,num_warps=4)\n        ref=a.float()@b.float()\n        torch.testing.assert_close(c,ref,rtol=5e-4,atol=2e-3)\n        errors.append(float((c-ref).abs().max()))\n    return {"applied":True,"capability":list(capability),"operation":"LoRA dot with FP32 accumulation",\n            "reference_test":"pass","maximum_errors":errors,"kernel_sha256":hashlib.sha256(path.read_bytes()).hexdigest()}\n', 'umilife/train_2gpu.py': '"""Notebook runtime: two-rank Unsloth QLoRA, explicit DDP loop and best adapter."""\nimport os\nos.environ.setdefault(\'UNSLOTH_COMPILE_DISABLE\',\'0\')\nos.environ.setdefault(\'UNSLOTH_RETURN_LOGITS\',\'0\')\nos.environ[\'UNSLOTH_RETURN_HIDDEN_STATES\']=\'1\'\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\',\'false\')\n\ndef main():\n    import argparse,json,hashlib,importlib.metadata\n    from pathlib import Path\n    import torch\n    local=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(local)\n    os.environ.setdefault(\'TORCHINDUCTOR_CACHE_DIR\',f\'/tmp/lifemate_inductor_rank{local}\')\n    os.environ.setdefault(\'TRITON_CACHE_DIR\',f\'/tmp/lifemate_triton_rank{local}\')\n    print(json.dumps({\'stage\':\'startup\',\'rank\':local,\'unsloth_compile_enabled\':os.environ[\'UNSLOTH_COMPILE_DISABLE\']==\'0\'}),flush=True)\n    from unsloth import FastLanguageModel\n    import torch.distributed as dist\n    from datetime import timedelta\n    from peft import set_peft_model_state_dict\n    from peft.utils.save_and_load import load_peft_weights\n    from umilife.core import digest,write_json,dumps\n    from umilife.models import TARGET_MODULES,seed_all\n    from umilife.train import Dataset,Collator,adapter_state\n    from umilife.prompt_policy import PROMPTS,POLICY_AUDIT\n    from umilife.ddp_engine import fit,agreement\n    from umilife.completion_loss import completion_loss\n    p=argparse.ArgumentParser()\n    for k in (\'stage\',\'data-dir\',\'output-dir\',\'model\',\'revision\'):p.add_argument(\'--\'+k,required=True)\n    p.add_argument(\'--train-batch-size\',type=int,default=4);p.add_argument(\'--eval-batch-size\',type=int,default=4)\n    p.add_argument(\'--eval-steps\',type=int,default=100)\n    p.add_argument(\'--init-from\',choices=(\'base\',),required=True)\n    p.add_argument(\'--epochs\',type=int,required=True)\n    a=p.parse_args()\n    assert a.train_batch_size>0 and a.eval_batch_size>0\n    assert int(os.environ[\'WORLD_SIZE\'])==2 and torch.cuda.device_count()==2\n    assert a.stage in (\'planner\',\'caller\')\n    assert a.init_from==\'base\',\'Independent local adapters only\'\n    assert a.epochs>0\n    dist.init_process_group(\'nccl\',timeout=timedelta(minutes=20),device_id=torch.device(\'cuda\',local))\n    seed_all(42)\n    data=Path(a.data_dir);out=Path(a.output_dir);target=out/\'adapters\'/a.stage\n    assert not (target/\'lineage.json\').exists(),\'Stage already complete\'\n    if target.exists():\n        import shutil;shutil.rmtree(target)\n    manifest=json.loads((data/\'manifest.json\').read_text())\n    prompt_sha=POLICY_AUDIT[\'static_prompts_sha256\']\n    assert manifest[\'train_episodes\']==1300 and manifest[\'validation_episodes\']==245\n    assert manifest[\'prompts_sha256\']==prompt_sha and manifest[\'rationale_authors\']==[\'Codex\']\n    for part in (\'train\',\'validation\'):\n        f=data/f\'{a.stage}.{part}.jsonl\';assert digest(f)==manifest[\'files_sha256\'][f.name]\n    dtype=torch.bfloat16 if torch.cuda.get_device_capability(local)[0]>=8 else torch.float16\n    model,tokenizer=FastLanguageModel.from_pretrained(model_name=a.model,revision=a.revision,\n        use_exact_model_name=True,max_seq_length=8192,dtype=dtype,load_in_4bit=True,\n        fast_inference=False,device_map={\'\':local},trust_remote_code=False)\n    model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,\n        target_modules=TARGET_MODULES,bias=\'none\',use_gradient_checkpointing=True,random_state=42)\n    def load_adapter(model,path):\n        result=set_peft_model_state_dict(model,load_peft_weights(str(path),device=\'cpu\'))\n        assert not result.unexpected_keys,result.unexpected_keys\n    initial=adapter_state(model)\n    assert all(\'lora_\' in n for n,v in model.named_parameters() if v.requires_grad)\n    FastLanguageModel.for_training(model)\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n    model.config.use_cache=False\n    def cached_dataset(part):\n        path=data/f\'{a.stage}.{part}.jsonl\'\n        cache=data/f\'{a.stage}.{part}.{digest(path)[:16]}.tokens.pt\'\n        if dist.get_rank()==0 and not cache.exists():\n            encoded=Dataset(path,tokenizer,8192)\n            temp=cache.with_suffix(\'.tmp\');torch.save({\'items\':encoded.items,\'stats\':encoded.stats},temp);temp.replace(cache)\n        dist.barrier()\n        payload=torch.load(cache,map_location=\'cpu\',weights_only=True)\n        encoded=Dataset.__new__(Dataset);encoded.items=payload[\'items\'];encoded.stats=payload[\'stats\']\n        return encoded\n    train=cached_dataset(\'train\');valid=cached_dataset(\'validation\')\n    assert train.stats[\'overlength\']==valid.stats[\'overlength\']==0,\'No training examples may be silently dropped\'\n    os.environ[\'TRAIN_RESUME_SIGNATURE\']=hashlib.sha256(json.dumps({\'manifest\':digest(data/\'manifest.json\'),\'stage\':a.stage,\'epochs\':a.epochs,\'batch\':a.train_batch_size,\'eval_batch\':a.eval_batch_size,\'eval_steps\':a.eval_steps,\'compile\':os.environ.get(\'UNSLOTH_COMPILE_DISABLE\'),\'loss_chunk\':os.environ.get(\'LIFEMATE_LOSS_CHUNK\',\'512\'),\'projection_checkpoint\':os.environ.get(\'LOSS_PROJECTION_CHECKPOINT\',\'0\'),\'model\':a.model,\'revision\':a.revision,\'init_from\':a.init_from},sort_keys=True).encode()).hexdigest()\n    def save_adapter(model,path):model.save_pretrained(path,safe_serialization=True)\n    result=fit(model,train,valid,Collator(tokenizer.pad_token_id),torch.device(\'cuda\',local),\n        out/\'checkpoints\'/a.stage,a.epochs,1e-5,\n        save_adapter,load_adapter,amp_dtype=dtype,train_batch_size=a.train_batch_size,\n        eval_batch_size=a.eval_batch_size,loss_function=completion_loss,\n        eval_steps=a.eval_steps)\n    final=adapter_state(model);assert final[\'finite\'] and final[\'sha256\']!=initial[\'sha256\']\n    agreement(final[\'sha256\'])\n    if dist.get_rank()==0:\n        save_adapter(model,target);tokenizer.save_pretrained(target)\n        history=result.pop(\'history\');write_json(target/\'training_history.json\',history)\n        record={**result,\'stage\':a.stage,\'model\':a.model,\'revision\':a.revision,\n            \'initialization\':a.init_from,\'smoke\':False,\'method\':\'Unsloth QLoRA independent local\',\'loss_implementation\':\'assistant-only token-mean CE; configurable projection checkpoint\' ,\'projection_checkpoint\':os.environ.get(\'LOSS_PROJECTION_CHECKPOINT\',\'0\')==\'1\',\'loss_chunk\':int(os.environ.get(\'LIFEMATE_LOSS_CHUNK\',\'512\')),\'initial_adapter\':initial,\'final_adapter\':final,\n            \'parent\':None,\n            \'global_adapter_sha256\':None,\n            \'dataset_manifest_sha256\':digest(data/\'manifest.json\'),\'prompt_sha256\':prompt_sha,\n            \'tokenization\':{\'train\':train.stats,\'validation\':valid.stats},\n            \'actual_records\':{\'train\':len(train),\'validation\':len(valid)},\n            \'peak_cuda_gb_rank0\':torch.cuda.max_memory_allocated()/2**30,\n            \'unsloth_compile_enabled\':os.environ.get(\'UNSLOTH_COMPILE_DISABLE\')==\'0\',\n            \'versions\':{k:importlib.metadata.version(k) for k in (\'torch\',\'transformers\',\'unsloth\',\'unsloth_zoo\',\'peft\',\'bitsandbytes\')}}\n        write_json(target/\'lineage.json\',record);print(dumps(record),flush=True)\n    dist.barrier();dist.destroy_process_group()\nif __name__==\'__main__\':main()\n', 'umilife/typescript_schema.py': '"""Compact TypeScript interfaces derived solely from the installed public catalog."""\nimport json\n\ndef literal(v):return json.dumps(v,ensure_ascii=False,separators=(\',\',\':\'))\ndef ts_type(p):\n    if \'enum\' in p:return \' | \'.join(literal(v) for v in p[\'enum\'])\n    t=p.get(\'type\')\n    if isinstance(t,list):return \' | \'.join(ts_type({**p,\'type\':v}) for v in t)\n    if t==\'array\':return \'(\'+ts_type(p.get(\'items\',{}))+\')[]\'\n    if t==\'object\':\n        fields=p.get(\'properties\')\n        if fields is None:return \'Record<string, unknown>\'\n        base=\'{ \'+ \'; \'.join(literal(k)+(\'\' if k in p.get(\'required\',[]) else \'?\')+\': \'+ts_type(v) for k,v in fields.items())+\' }\'\n        if p.get(\'x-one-of\'):\n            alt=[\'{ \'+\'; \'.join(literal(k)+\': \'+ts_type(fields[k]) for k in group)+\' }\' for group in p[\'x-one-of\']]\n            return base+\' & (\'+\' | \'.join(alt)+\')\'\n        return base\n    return {\'string\':\'string\',\'integer\':\'number\',\'number\':\'number\',\'boolean\':\'boolean\',\'null\':\'null\'}.get(t,\'unknown\')\n\ndef metadata(p,path=\'\',detailed=False):\n    result=[]\n    for k,v in p.items():\n        if k in (\'properties\',\'items\',\'required\',\'type\',\'enum\',\'x-one-of\'):continue\n        if k==\'description\' and not detailed:continue\n        result.append((path+\'.\' if path else \'\')+k+\'=\'+literal(v))\n    if p.get(\'type\')==\'integer\':result.append(path+\':integer\')\n    for k,v in p.get(\'properties\',{}).items():result.extend(metadata(v,(path+\'.\' if path else \'\')+k,detailed))\n    if p.get(\'items\'):result.extend(metadata(p[\'items\'],path+\'[]\',detailed))\n    return result\n\ndef typescript_catalog(specs,detailed=False):\n    lines=[\'interface InstalledTools {\']\n    for name,s in sorted(specs.items()):\n        comment=\'; \'.join([s.get(\'kind\',\'\'),s.get(\'description\',\'\'),*metadata(s[\'parameters\'],detailed=detailed)])\n        if s.get(\'returns\') and detailed:comment+=\'; returns=\'+str(s[\'returns\'])\n        comment=comment.replace(\'*/\',\'* /\').replace(\'\\n\',\' \')\n        lines.append(\'  /** \'+comment+\' */\')\n        lines.append(\'  \'+literal(name)+\'(args: \'+ts_type(s[\'parameters\'])+\'): unknown;\')\n    return \'\\n\'.join(lines+[\'}\'])\n', 'zero_action_train_labels.json': '{\n  "author": "Codex",\n  "scope": "Training-only manually reviewed no-action terminal targets; never loaded for validation target construction or model messages.",\n  "labels": {\n    "train_0036": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0038": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0155": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0432": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0446": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0460": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0532": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0606": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0655": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0685": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0686": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0740": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0825": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0859": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0920": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0926": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0929": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0993": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0997": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1067": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1114": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1135": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1167": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1185": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1192": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1280": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1286": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_1706": {\n      "thought": "The latest request cancels or explicitly withholds the operation. Leave existing state unchanged and end this turn.",\n      "next": "summarizer"\n    },\n    "train_0057": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0431": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0445": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0459": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0531": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0565": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0602": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0738": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0797": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0824": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0858": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0861": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_1452": {\n      "thought": "The user is acknowledging, chatting or describing a completed activity rather than requesting a new tool operation. End without changing state.",\n      "next": "summarizer"\n    },\n    "train_0082": {\n      "thought": "The prior forecast says no rain tomorrow. The condition for moving the run and sending the notification is false; do neither.",\n      "next": "summarizer"\n    },\n    "train_0137": {\n      "thought": "The prior forecast for tomorrow reports no rain. Leave the event unchanged and do not send the conditional SMS.",\n      "next": "summarizer"\n    },\n    "train_0266": {\n      "thought": "The existing forecast for tomorrow reports rain=false. The requested reschedule and SMS are conditional on rain, so end without changes.",\n      "next": "summarizer"\n    },\n    "train_0125": {\n      "thought": "The observed SJC buying price is 130,800,000 VND, not below 129,000,000. Do not create the conditional reminder.",\n      "next": "summarizer"\n    },\n    "train_0232": {\n      "thought": "The observed SJC buying price is 124,900,000 VND, not below 123,000,000. Do not create the conditional reminder.",\n      "next": "summarizer"\n    },\n    "train_0394": {\n      "thought": "The observed SJC buying price is 119,000,000 VND, not below 118,000,000. Leave reminders unchanged.",\n      "next": "summarizer"\n    },\n    "train_0327": {\n      "thought": "The previous gold-price result already contains the requested SJC buying price,128,900,000 VND. Answer from that history without another lookup.",\n      "next": "summarizer"\n    },\n    "train_1358": {\n      "thought": "The history confirms the electricity transfer to Trang was already completed. Reuse that confirmation and do not transfer again.",\n      "next": "summarizer"\n    },\n    "train_1360": {\n      "thought": "The observed reward requires 6,000 points but the account has only 3,000. The condition for redemption is false; do not redeem a different reward.",\n      "next": "summarizer"\n    },\n    "train_1367": {\n      "thought": "The most recent wallet balance is 150,000 VND, below the requested 200,000 threshold. Do not perform another top-up.",\n      "next": "summarizer"\n    },\n    "train_1446": {\n      "thought": "The history confirms the requested lesson is already scheduled. The request is conditional on it not being scheduled; do not create a duplicate.",\n      "next": "summarizer"\n    },\n    "train_1453": {\n      "thought": "The user says the film was already added and explicitly asks not to add it again. End without another write.",\n      "next": "summarizer"\n    },\n    "train_1494": {\n      "thought": "The prior weather result reports no rain, so the pickup condition in the message is false. No school-message lookup is needed for this conditional request.",\n      "next": "summarizer"\n    },\n    "train_1593": {\n      "thought": "Sending 20,000,000 from the observed 35,200,000 balance leaves 15,200,000, below the 16,500,000 tuition commitment. Do not open the conditional deposit.",\n      "next": "summarizer"\n    },\n    "train_0019": {\n      "thought": "The installed tools do not support hotel reservations. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0043": {\n      "thought": "The installed tools do not support ride booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0248": {\n      "thought": "The installed tools do not support ride booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0428": {\n      "thought": "The installed tools do not support train-ticket booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0434": {\n      "thought": "The installed tools do not support retrieving the entire requested book text. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0447": {\n      "thought": "The installed tools do not support guessing another person’s banking password. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0458": {\n      "thought": "The installed tools do not support wallet transfers. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0496": {\n      "thought": "The installed tools do not support music album playback. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0499": {\n      "thought": "The installed tools do not support covert location tracking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0530": {\n      "thought": "The installed tools do not support money transfers. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0562": {\n      "thought": "The installed tools do not support music album playback. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0600": {\n      "thought": "The installed tools do not support money transfers. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0601": {\n      "thought": "The installed tools do not support medical appointment booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0687": {\n      "thought": "The installed tools do not support booking the requested licensing appointment. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0735": {\n      "thought": "The installed tools do not support weather lookup. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0736": {\n      "thought": "The installed tools do not support course enrollment. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0788": {\n      "thought": "The installed tools do not support hotel-room reservations. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0798": {\n      "thought": "The installed tools do not support international money transfers. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0856": {\n      "thought": "The installed tools do not support flight booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0863": {\n      "thought": "The installed tools do not support camera access and face identification. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0938": {\n      "thought": "The installed tools do not support joining an online meeting. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0956": {\n      "thought": "The installed tools do not support vehicle-fine lookup. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0982": {\n      "thought": "The installed tools do not support arming a security alarm. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0990": {\n      "thought": "The installed tools do not support reading SMS messages. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1004": {\n      "thought": "The installed tools do not support music playback control. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1032": {\n      "thought": "The installed tools do not support sports-results lookup. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1116": {\n      "thought": "The installed tools do not support assigning household chores. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1143": {\n      "thought": "The installed tools do not support loyalty-points lookup. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1156": {\n      "thought": "The installed tools do not support taxi booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1161": {\n      "thought": "The installed tools do not support email replies. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1172": {\n      "thought": "The installed tools do not support flight check-in. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1242": {\n      "thought": "The installed tools do not support reminder cancellation and creation. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1299": {\n      "thought": "The installed tools do not support file sharing. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1316": {\n      "thought": "The installed tools do not support currency conversion. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1386": {\n      "thought": "The installed tools do not support medical appointment booking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1436": {\n      "thought": "The installed tools do not support adding to a video watchlist. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1513": {\n      "thought": "The installed tools do not support grocery ordering. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1518": {\n      "thought": "The installed tools do not support file sharing with view-only permissions. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1538": {\n      "thought": "The installed tools do not support the requested messaging or phone call. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1555": {\n      "thought": "The installed tools do not support starting a dishwasher cycle. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1582": {\n      "thought": "The installed tools do not support opening and reading the invoice. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1587": {\n      "thought": "The installed tools do not support driving-route lookup. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1588": {\n      "thought": "The installed tools do not support pantry-consumption tracking. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_1655": {\n      "thought": "The installed tools do not support library reservations. Stop without substituting an unrelated operation or inventing a result.",\n      "next": "give_up"\n    },\n    "train_0688": {\n      "thought": "A negative brightness percentage is outside the supported tool range. Do not silently replace it with a different brightness.",\n      "next": "give_up"\n    }\n  }\n}\n'}
SOURCE_SHA='d5296eea972d558a7d9f3aa2b9ccf2381dedf8eb4932368f22e7f7a71bce1787'
assert hashlib.sha256(json.dumps(PAYLOAD,sort_keys=True).encode()).hexdigest()==SOURCE_SHA
for name,content in PAYLOAD.items():
 p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
config={'source_sha':SOURCE_SHA,'model':MODEL,'revision':REVISION,'epochs':EPOCHS,'micro_batch':4,'world_size':2,'stages':list(STAGES),'compile_enabled':True,'loss_chunk':LOSS_CHUNK,'projection_checkpoint':PROJECTION_CHECKPOINT,'eval_steps':EVAL_STEPS,'save_steps':SAVE_STEPS,'initialization':'independent_base'}
if (WORK/'run_config.json').exists():assert json.loads((WORK/'run_config.json').read_text())==config,'Incompatible resume directory'
else:(WORK/'run_config.json').write_text(json.dumps(config,indent=2))


In [ ]:
sys.path.insert(0,str(SOURCE))
from umilife.paths import find_data_root
DATA_ROOT,archive=find_data_root('/kaggle/input')
if DATA_ROOT is None:
 assert archive is not None,'Attach the competition dataset'
 dest=Path('/tmp/evidence_v3_data');dest.mkdir(exist_ok=True)
 with zipfile.ZipFile(archive) as z:
  assert all((dest/m.filename).resolve().is_relative_to(dest.resolve()) for m in z.infolist())
  z.extractall(dest)
 DATA_ROOT,_=find_data_root(dest)
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv==0.9.10'],check=True)
UV=[sys.executable,'-m','uv'];os.environ['UV_NATIVE_TLS']='true'
venv=Path('/tmp/evidence_v3_unsloth');PYTHON=str(venv/'bin/python')
if not Path(PYTHON).exists():subprocess.run([*UV,'venv','--python','3.12','--seed',str(venv)],check=True)
subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',PYTHON,str(SOURCE/'requirements_train_2gpu.lock')],check=True)
subprocess.run([*UV,'pip','check','--python',PYTHON],check=True)
ENV=os.environ.copy()
for key in ('PYTHONPATH','PYTHONHOME'):ENV.pop(key,None)
ENV.update(CUDA_VISIBLE_DEVICES='0,1',PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='2',
 UNSLOTH_COMPILE_DISABLE='0',LIFEMATE_LOSS_CHUNK=str(LOSS_CHUNK),LOSS_PROJECTION_CHECKPOINT=str(int(PROJECTION_CHECKPOINT)),INITIAL_EVAL='0',VERIFY_RESTORED_EVAL='0',SAVE_STEPS=str(SAVE_STEPS),TRAIN_DEADLINE=str(DEADLINE),HF_HOME='/tmp/lifemate_hf_cache')
purelib=Path(subprocess.check_output([PYTHON,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
ENV['LD_LIBRARY_PATH']=':'.join(str(p) for p in (purelib/'nvidia').glob('*/lib'))+':'+ENV.get('LD_LIBRARY_PATH','')
def execute(module,args,ddp=False,env=None,check=True):
 command=[PYTHON,'-u','-m']+(['torch.distributed.run','--standalone','--nproc_per_node=2','--module',module] if ddp else [module])
 return subprocess.run(command+list(map(str,args)),cwd=SOURCE,env=env or ENV,check=check)
# No synthetic GPU preflight or model smoke. Verify real training startup in the log.
for key in ('SMOKE_PAUSE_STEP',):ENV.pop(key,None)
DATA=WORK/'prepared'
if not (DATA/'manifest.json').exists():
 execute('umilife.prepare_2gpu',['--data-root',DATA_ROOT,'--output',DATA,'--split-file',SOURCE/'split.txt','--model',MODEL,'--revision',REVISION])
m=json.loads((DATA/'manifest.json').read_text());assert m['train_episodes']==1300 and m['validation_episodes']==245
assert m['split_sha256']==hashlib.sha256((SOURCE/'split.txt').read_bytes()).hexdigest()
print('Data:',m['counts']);print('Annotations:',m['rationale_provenance']);print('Quarantined train episodes:',m['train_quarantined_episodes']);print('Audit:',DATA/'data_audit.json')


In [ ]:
# Build a separate vLLM0.9.2 environment only if generation evaluation is requested.
if RUN_NODE_EVAL:
 infer_venv=Path('/tmp/evidence_v3_vllm');INFER_PYTHON=str(infer_venv/'bin/python')
 if not Path(INFER_PYTHON).exists():subprocess.run([*UV,'venv','--python','3.12','--seed',str(infer_venv)],check=True)
 subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',INFER_PYTHON,str(SOURCE/'requirements_infer_2gpu.lock')],check=True)
 subprocess.run([*UV,'pip','check','--python',INFER_PYTHON],check=True)
 INFER_ENV={k:v for k,v in ENV.items() if not k.startswith('UNSLOTH_')}
 INFER_ENV.update(VLLM_USE_V1='0',VLLM_ATTENTION_BACKEND='XFORMERS',VLLM_WORKER_MULTIPROC_METHOD='spawn',VLLM_USAGE_STATS_ENABLED='0',DO_NOT_TRACK='1')
 infer_purelib=Path(subprocess.check_output([INFER_PYTHON,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
 INFER_ENV['LD_LIBRARY_PATH']=':'.join(str(p) for p in (infer_purelib/'nvidia').glob('*/lib'))+':'+os.environ.get('LD_LIBRARY_PATH','')


In [ ]:
OUTPUT=WORK/'training'
recommendations={}
for stage in STAGES:
 target=OUTPUT/'adapters'/stage
 done=target/'lineage.json'
 if done.exists():
  record=json.loads(done.read_text())
  assert record['dataset_manifest_sha256']==hashlib.sha256((DATA/'manifest.json').read_bytes()).hexdigest()
  assert record['stage']==stage and record['initialization']=='base' and not record['smoke']
  print('Already complete:',stage)
 else:
  print('TRAINING:',stage,'2 GPUs, microbatch4, independent base initialization',flush=True)
  args=['--stage',stage,'--data-dir',DATA,'--output-dir',OUTPUT,'--model',MODEL,'--revision',REVISION,
        '--init-from','base','--epochs',EPOCHS[stage],'--train-batch-size',4,'--eval-batch-size',4,'--eval-steps',EVAL_STEPS]
  result=execute('umilife.train_2gpu',args,ddp=True,check=False)
  assert result.returncode==0,'Paused or failed: preserve WORK, inspect logs/latest checkpoint, then resume the same run.'
 if RUN_NODE_EVAL:
  print('NODE GENERATION EVAL:',stage,'CE-best vs last; not a full workflow score',flush=True)
  report_dir=WORK/'node_eval'/stage
  args=['--stage',stage,'--prepared',DATA,'--adapter',target,'--checkpoints',OUTPUT/'checkpoints'/stage,
        '--data-root',DATA_ROOT,'--model',MODEL,'--revision',REVISION,'--output',report_dir]
  subprocess.run([INFER_PYTHON,'-u','-m','umilife.node_eval',*map(str,args)],cwd=SOURCE,env=INFER_ENV,check=True)
  recommendations[stage]=json.loads((report_dir/'report.json').read_text())
  (WORK/'node_checkpoint_recommendations.json').write_text(json.dumps(recommendations,ensure_ascii=False,indent=2))
selected=WORK/'selected_adapters'
if RUN_NODE_EVAL:
 roles=sorted(p.name for p in selected.iterdir() if p.is_dir() and (p/'lineage.json').exists())
 raw_sha=hashlib.sha256((DATA/'manifest.json').read_bytes()).hexdigest()
 contract={'canonical_manifest_sha256':hashlib.sha256(json.dumps(m,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest(),'raw_manifest_sha256':{r:raw_sha for r in roles}}
 (selected/'data_contract.json').write_text(json.dumps(contract,indent=2))
 files={str(p.relative_to(selected)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(selected.rglob('*')) if p.is_file() and p.name!='checkpoint_manifest.json'}
 (selected/'checkpoint_manifest.json').write_text(json.dumps({'roles':roles,'model':MODEL,'revision':REVISION,'prompt_sha256':m['prompts_sha256'],'files':files},indent=2))
 print('Exported selected adapters:',selected)
print('Preserve entire directory:',WORK)
